
# Notebook 06 — Final Project Synthesis and Source Verification

## Kenya Electoral Participation Analytics

### Objective

Strengthen the reliability of the final project findings by
attempting independent reconciliation of the 2022 county-level
presidential election figures against official IEBC publications.

### Verification principles

1. Preserve all original ELOG source records.
2. Preserve the existing validated analytical dataset.
3. Use official IEBC declarations as the preferred reference.
4. Treat third-party transcriptions as secondary evidence.
5. Record all differences, including small numerical discrepancies.
6. Do not automatically replace figures or change validation statuses.
7. Reassess published findings if material discrepancies are confirmed.

### Scope

- Election: 9 August 2022 general election
- Geography: Kenya's 47 counties
- Primary fields: registered voters and valid votes
- Additional fields, where available: rejected ballots and ballots cast

### Expected outputs

- Official 2022 source inventory
- County-level source reconciliation
- Validation and discrepancy register
- Assessment of implications for the final analytical findings


### Load and audit the existing 2022 dataset

In [3]:

# ================================================================
# NOTEBOOK 06 — CELL 01
# ESTABLISH WORKSPACE AND AUDIT 2022 ANALYTICAL DATA
# ================================================================
# Read-only cell: no existing files are modified.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Locate project root from the notebook directory.
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "Interim"
DOCS_DIR = PROJECT_ROOT / "documentation"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

analytical_path = (
    PROCESSED_DIR / "kenya_county_election_analytics.csv"
)

assert analytical_path.is_file(), (
    f"Analytical dataset not found: {analytical_path}"
)

# Load the validated analytical dataset.
elections_df = pd.read_csv(analytical_path)

# Isolate the 2022 election.
election_2022 = elections_df.loc[
    elections_df["election_id"].eq("2022_GENERAL")
].copy()

# Verify county coverage and unique keys.
assert len(election_2022) == 47
assert election_2022["county_key"].nunique() == 47
assert not election_2022["county_key"].duplicated().any()

# Confirm that the expected fields exist.
required_columns = [
    "county",
    "county_key",
    "analytical_registered_voters",
    "analytical_valid_votes",
    "analytical_ballots_cast",
    "valid_vote_participation_pct",
    "validation_status"
]

assert set(required_columns).issubset(
    election_2022.columns
)

# Build a source-verification baseline.
baseline_2022 = election_2022[
    required_columns
].sort_values("county_key").reset_index(drop=True)

print("NOTEBOOK 06 — 2022 SOURCE-VERIFICATION BASELINE")
print("=" * 65)

print("Project root:", PROJECT_ROOT)
print("County records:", len(baseline_2022))
print("Unique counties:", baseline_2022["county_key"].nunique())

print(
    "Total registered voters:",
    f"{baseline_2022['analytical_registered_voters'].sum():,}"
)

print(
    "Total valid votes:",
    f"{baseline_2022['analytical_valid_votes'].sum():,}"
)

print(
    "Missing registered-voter figures:",
    baseline_2022["analytical_registered_voters"].isna().sum()
)

print(
    "Missing valid-vote figures:",
    baseline_2022["analytical_valid_votes"].isna().sum()
)

print("\nVALIDATION STATUS DISTRIBUTION")
print(
    baseline_2022["validation_status"].value_counts(
        dropna=False
    ).to_string()
)

print("\n2022 COUNTY RECORDS — FIRST 10")
display(baseline_2022.head(10))

print("\nCELL 01 BASELINE AUDIT: PASSED")


NOTEBOOK 06 — 2022 SOURCE-VERIFICATION BASELINE
Project root: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project
County records: 47
Unique counties: 47
Total registered voters: 22,110,015
Total valid votes: 14,207,149.0
Missing registered-voter figures: 0
Missing valid-vote figures: 0

VALIDATION STATUS DISTRIBUTION
validation_status
COUNTY_SOURCE_UNVERIFIED    47

2022 COUNTY RECORDS — FIRST 10


,county,county_key,analytical_registered_voters,analytical_valid_votes,analytical_ballots_cast,valid_vote_participation_pct,validation_status
0,Baringo,BARINGO,281107,217091.0,NaN,77.23,COUNTY_SOURCE_UNVERIFIED
1,Bomet,BOMET,377023,299606.0,NaN,79.47,COUNTY_SOURCE_UNVERIFIED
2,Bungoma,BUNGOMA,646612,405151.0,NaN,62.66,COUNTY_SOURCE_UNVERIFIED
3,Busia,BUSIA,416818,276730.0,NaN,66.39,COUNTY_SOURCE_UNVERIFIED
4,Elgeyo-Marakwet,ELGEYOMARAKWET,213904,165224.0,NaN,77.24,COUNTY_SOURCE_UNVERIFIED
5,Embu,EMBU,334684,221048.0,NaN,66.05,COUNTY_SOURCE_UNVERIFIED
6,Garissa,GARISSA,201513,109960.0,NaN,54.57,COUNTY_SOURCE_UNVERIFIED
7,Homa Bay,HOMABAY,551111,404112.0,NaN,73.33,COUNTY_SOURCE_UNVERIFIED
8,Isiolo,ISIOLO,89535,58978.0,NaN,65.87,COUNTY_SOURCE_UNVERIFIED
9,Kajiado,KAJIADO,463389,308624.0,NaN,66.60,COUNTY_SOURCE_UNVERIFIED



CELL 01 BASELINE AUDIT: PASSED


### Create the official-source inventory
This cell records the sources we intend to investigate. It does not download data or assume that a source has already been authenticated.

Before importing any figures, we need to establish their provenance.

Our preferred source is the IEBC's 2022 presidential election Form 34C, which consolidates presidential election results.

We should distinguish between:
- Primary evidence: The official IEBC declaration or an authenticated reproduction.
- Secondary evidence: A third-party table transcribing the official declaration.
- Existing analytical evidence: Our ELOG-derived county dataset.
A secondary transcription can help us locate discrepancies, but it should not automatically change a county's verification status to COUNTY_RECONCILED.

In [7]:

# ================================================================
# NOTEBOOK 06 — CELL 02
# 2022 OFFICIAL-SOURCE INVENTORY
# ================================================================
# Purpose:
# 1. Register potential 2022 verification sources.
# 2. Distinguish primary and secondary evidence.
# 3. Track verification progress.
# 4. Preserve the existing analytical dataset.
#
# This cell creates a source inventory only.
# ================================================================

import pandas as pd

source_inventory_2022 = pd.DataFrame([
    {
        "source_id": "IEBC_2022_FORM34C",
        "organisation": "IEBC",
        "document": "2022 Presidential Election Form 34C",
        "source_type": "Primary official declaration",
        "fields_required": (
            "County, registered voters, valid votes, "
            "rejected ballots"
        ),
        "verification_status": "NOT_YET_AUTHENTICATED",
        "extraction_status": "NOT_STARTED",
        "notes": (
            "Preferred reconciliation source. "
            "Confirm document authenticity and "
            "county-level field definitions."
        )
    },
    {
        "source_id": "2022_SECONDARY_TRANSCRIPTION",
        "organisation": "Third-party publisher",
        "document": (
            "County-level transcription of "
            "2022 presidential results"
        ),
        "source_type": "Secondary transcription",
        "fields_required": (
            "County, registered voters, valid votes, "
            "rejected ballots"
        ),
        "verification_status": "NOT_YET_VERIFIED",
        "extraction_status": "NOT_STARTED",
        "notes": (
            "Use as a cross-check. Do not treat "
            "as equivalent to an authenticated "
            "official declaration."
        )
    },
    {
        "source_id": "ELOG_2022_BASELINE",
        "organisation": "ELOG",
        "document": "Existing 2022 county analytical records",
        "source_type": "Existing primary analytical input",
        "fields_required": (
            "County, registered voters, valid votes"
        ),
        "verification_status": "COUNTY_SOURCE_UNVERIFIED",
        "extraction_status": "COMPLETE",
        "notes": (
            "47 counties; 22,110,015 registered "
            "voters; 14,207,149 valid votes."
        )
    }
])

print("2022 SOURCE INVENTORY")
print("=" * 65)

display(source_inventory_2022)

print("\nSOURCE TYPES")
print(
    source_inventory_2022[
        "source_type"
    ].value_counts().to_string()
)

print("\nCELL 02 SOURCE INVENTORY: PASSED")


2022 SOURCE INVENTORY


,source_id,organisation,document,source_type,fields_required,verification_status,extraction_status,notes
0,IEBC_2022_FORM34C,IEBC,2022 Presidential Election Form 34C,Primary official declaration,"County, registered voters, valid votes, reject...",NOT_YET_AUTHENTICATED,NOT_STARTED,Preferred reconciliation source. Confirm docum...
1,2022_SECONDARY_TRANSCRIPTION,Third-party publisher,County-level transcription of 2022 presidentia...,Secondary transcription,"County, registered voters, valid votes, reject...",NOT_YET_VERIFIED,NOT_STARTED,Use as a cross-check. Do not treat as equivale...
2,ELOG_2022_BASELINE,ELOG,Existing 2022 county analytical records,Existing primary analytical input,"County, registered voters, valid votes",COUNTY_SOURCE_UNVERIFIED,COMPLETE,"47 counties; 22,110,015 registered voters; 14,..."



SOURCE TYPES
source_type
Primary official declaration         1
Secondary transcription              1
Existing primary analytical input    1

CELL 02 SOURCE INVENTORY: PASSED


### Inspect the downloaded CSV
This cell will:
1. Load the original CSV without modifying it.
2. Display its dimensions and column names.
3. Examine sample records and missing values.
4. Identify potential election, county and vote-related columns.

In [10]:

# ================================================================
# NOTEBOOK 06 — CELL 03
# INSPECT TAKWIMU 2022 ELECTION RESULTS DATASET
# ================================================================
# Purpose:
# 1. Load the original Takwimu CSV.
# 2. Examine its structure and available fields.
# 3. Identify potential presidential election records.
# 4. Assess suitability for 2022 reconciliation.
#
# Read-only inspection: no source files are modified.
# ================================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------
# STEP 1: LOCATE THE SOURCE FILE
# ------------------------------------------------

takwimu_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "reference"
    / "results-2022-general-election-by-seat.csv"
)

assert takwimu_path.is_file(), (
    f"Takwimu CSV not found: {takwimu_path}"
)

# ------------------------------------------------
# STEP 2: LOAD THE CSV
# ------------------------------------------------

takwimu_df = pd.read_csv(
    takwimu_path,
    low_memory=False
)

print("TAKWIMU 2022 — SOURCE DATA INSPECTION")
print("=" * 65)

print("File:", takwimu_path.name)
print("Rows:", len(takwimu_df))
print("Columns:", len(takwimu_df.columns))

# ------------------------------------------------
# STEP 3: INSPECT COLUMN STRUCTURE
# ------------------------------------------------

print("\nCOLUMN NAMES")
print("=" * 65)

for i, col in enumerate(takwimu_df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nCOLUMN DATA TYPES")
display(
    takwimu_df.dtypes.to_frame("data_type")
)

# ------------------------------------------------
# STEP 4: INSPECT SAMPLE RECORDS
# ------------------------------------------------

print("\nFIRST 10 RECORDS")
display(takwimu_df.head(10))

print("\nLAST 5 RECORDS")
display(takwimu_df.tail(5))

# ------------------------------------------------
# STEP 5: CHECK MISSING VALUES
# ------------------------------------------------

missing_summary = pd.DataFrame({
    "column": takwimu_df.columns,
    "missing_count": takwimu_df.isna().sum().values,
    "missing_pct": (
        takwimu_df.isna().mean().values * 100
    ).round(2)
})

print("\nMISSING VALUE SUMMARY")
display(missing_summary)

# ------------------------------------------------
# STEP 6: IDENTIFY RELEVANT FIELDS
# ------------------------------------------------

keywords = [
    "county",
    "seat",
    "position",
    "president",
    "registered",
    "valid",
    "rejected",
    "votes",
    "candidate",
    "office"
]

potential_columns = [
    col for col in takwimu_df.columns
    if any(
        keyword in col.lower()
        for keyword in keywords
    )
]

print("\nPOTENTIALLY RELEVANT COLUMNS")
print("=" * 65)

for col in potential_columns:
    print("-", col)

print("\nCELL 03 SOURCE INSPECTION: COMPLETE")


TAKWIMU 2022 — SOURCE DATA INSPECTION
File: results-2022-general-election-by-seat.csv
Rows: 2077
Columns: 20

COLUMN NAMES
01. Seat
02. County code
03. County
04. Constituency code
05. Constituency
06. Ward code
07. Ward
08. Candidate
09. Party as printed
10. Abbreviation
11. Votes
12. Share of valid votes
13. Registered voters
14. Valid votes
15. Rejected ballots
16. Turnout
17. Deputy governor
18. Result
19. Source
20. Note

COLUMN DATA TYPES


,data_type
Seat,object
County code,float64
County,object
Constituency code,float64
Constituency,object
Ward code,float64
Ward,object
Candidate,object
Party as printed,object
Abbreviation,object



FIRST 10 RECORDS


,Seat,County code,County,Constituency code,Constituency,Ward code,Ward,Candidate,Party as printed,Abbreviation,Votes,Share of valid votes,Registered voters,Valid votes,Rejected ballots,Turnout,Deputy governor,Result,Source,Note
0,President,1.0,Mombasa,NaN,NaN,NaN,NaN,Odinga Raila,NaN,NaN,161015.0,58.07%,642362.0,277301.0,3812.0,43.76%,NaN,NaN,"Form 34C, 15 August 2022",NaN
1,President,1.0,Mombasa,NaN,NaN,NaN,NaN,Ruto William Samoei,NaN,NaN,113700.0,41.00%,642362.0,277301.0,3812.0,43.76%,NaN,NaN,"Form 34C, 15 August 2022",NaN
2,President,1.0,Mombasa,NaN,NaN,NaN,NaN,Waihiga David Mwaure,NaN,NaN,482.0,0.17%,642362.0,277301.0,3812.0,43.76%,NaN,NaN,"Form 34C, 15 August 2022",NaN
3,President,1.0,Mombasa,NaN,NaN,NaN,NaN,Wajackoyah George Luchiri,NaN,NaN,2104.0,0.76%,642362.0,277301.0,3812.0,43.76%,NaN,NaN,"Form 34C, 15 August 2022",NaN
4,President,2.0,Kwale,NaN,NaN,NaN,NaN,Odinga Raila,NaN,NaN,125541.0,70.13%,328316.0,179009.0,1359.0,54.94%,NaN,NaN,"Form 34C, 15 August 2022",NaN
5,President,2.0,Kwale,NaN,NaN,NaN,NaN,Ruto William Samoei,NaN,NaN,51918.0,29.00%,328316.0,179009.0,1359.0,54.94%,NaN,NaN,"Form 34C, 15 August 2022",NaN
6,President,2.0,Kwale,NaN,NaN,NaN,NaN,Waihiga David Mwaure,NaN,NaN,413.0,0.23%,328316.0,179009.0,1359.0,54.94%,NaN,NaN,"Form 34C, 15 August 2022",NaN
7,President,2.0,Kwale,NaN,NaN,NaN,NaN,Wajackoyah George Luchiri,NaN,NaN,1137.0,0.64%,328316.0,179009.0,1359.0,54.94%,NaN,NaN,"Form 34C, 15 August 2022",NaN
8,President,3.0,Kilifi,NaN,NaN,NaN,NaN,Odinga Raila,NaN,NaN,204536.0,71.64%,588842.0,285496.0,3191.0,49.03%,NaN,NaN,"Form 34C, 15 August 2022","On the form the four candidates add up to 285,..."
9,President,3.0,Kilifi,NaN,NaN,NaN,NaN,Ruto William Samoei,NaN,NaN,77331.0,27.09%,588842.0,285496.0,3191.0,49.03%,NaN,NaN,"Form 34C, 15 August 2022","On the form the four candidates add up to 285,..."



LAST 5 RECORDS


,Seat,County code,County,Constituency code,Constituency,Ward code,Ward,Candidate,Party as printed,Abbreviation,Votes,Share of valid votes,Registered voters,Valid votes,Rejected ballots,Turnout,Deputy governor,Result,Source,Note
2072,Member of county assembly,47.0,Nairobi City,290.0,Mathare,1446.0,Mabatini,Odalo Wilfred Oluoch,Orange Democratic Movement,ODM,5192.0,NaN,NaN,NaN,NaN,NaN,NaN,Elected,Gazette Notice No. 9956,NaN
2073,Member of county assembly,47.0,Nairobi City,290.0,Mathare,1447.0,Huruma,Owera Oluoch Peter,Orange Democratic Movement,ODM,5641.0,NaN,NaN,NaN,NaN,NaN,NaN,Elected,Gazette Notice No. 9956,NaN
2074,Member of county assembly,47.0,Nairobi City,290.0,Mathare,1448.0,Ngei,Onyango Redson Otieno,Orange Democratic Movement,ODM,3683.0,NaN,NaN,NaN,NaN,NaN,NaN,Elected,Gazette Notice No. 9956,NaN
2075,Member of county assembly,47.0,Nairobi City,290.0,Mathare,1449.0,Mlango Kubwa,Kavaya Susan Makungu,United Democratic Alliance,UDA,5790.0,NaN,NaN,NaN,NaN,NaN,NaN,Elected,Gazette Notice No. 9956,NaN
2076,Member of county assembly,47.0,Nairobi City,290.0,Mathare,1450.0,Kiamaiko,Ndung'u Joseph Karanja,United Democratic Alliance,UDA,4251.0,NaN,NaN,NaN,NaN,NaN,NaN,Elected,Gazette Notice No. 9956,NaN



MISSING VALUE SUMMARY


,column,missing_count,missing_pct
0,Seat,0,0.00
1,County code,8,0.39
2,County,0,0.00
3,Constituency code,337,16.23
4,Constituency,337,16.23
5,Ward code,627,30.19
6,Ward,627,30.19
7,Candidate,5,0.24
8,Party as printed,201,9.68
9,Abbreviation,203,9.77



POTENTIALLY RELEVANT COLUMNS
- Seat
- County code
- County
- Candidate
- Votes
- Share of valid votes
- Registered voters
- Valid votes
- Rejected ballots

CELL 03 SOURCE INSPECTION: COMPLETE


### Validate presidential records
Before creating the county reconciliation table, we'll check whether the presidential records provide exactly four candidate rows per county, whether repeated county totals agree, and whether candidate-vote sums match reported valid votes.

This is a read-only inspection.

In [13]:

# ================================================================
# NOTEBOOK 06 — CELL 04
# VALIDATE TAKWIMU 2022 PRESIDENTIAL RECORDS
# ================================================================
# Purpose:
# 1. Isolate presidential election records.
# 2. Check county coverage and candidate counts.
# 3. Validate repeated county-level totals.
# 4. Compare candidate-vote sums with reported valid votes.
# 5. Inspect source notes and potential exceptions.
#
# No files are created or modified.
# ================================================================

# ------------------------------------------------
# STEP 1: FILTER PRESIDENTIAL RECORDS
# ------------------------------------------------

presidential_2022 = takwimu_df.loc[
    takwimu_df["Seat"].astype(str).str.strip().eq("President")
].copy()

print("2022 PRESIDENTIAL RECORDS")
print("=" * 65)

print("Presidential rows:", len(presidential_2022))
print(
    "Distinct county labels:",
    presidential_2022["County"].nunique()
)

print("\nCOUNTY LABELS")
print(
    sorted(
        presidential_2022["County"]
        .dropna()
        .astype(str)
        .unique()
    )
)

# ------------------------------------------------
# STEP 2: INSPECT CANDIDATE COUNTS BY COUNTY
# ------------------------------------------------

candidate_counts = (
    presidential_2022
    .groupby("County", dropna=False)
    .agg(
        candidate_rows=("Candidate", "size"),
        unique_candidates=("Candidate", "nunique"),
        registered_values=("Registered voters", "nunique"),
        valid_vote_values=("Valid votes", "nunique"),
        rejected_values=("Rejected ballots", "nunique"),
        source_values=("Source", "nunique")
    )
    .reset_index()
)

print("\nCANDIDATE AND REPEATED-TOTAL CHECKS")
display(candidate_counts)

# ------------------------------------------------
# STEP 3: IDENTIFY INCONSISTENT COUNTY RECORDS
# ------------------------------------------------

inconsistent_counties = candidate_counts.loc[
    candidate_counts["candidate_rows"].ne(4)
    | candidate_counts["unique_candidates"].ne(4)
    | candidate_counts["registered_values"].ne(1)
    | candidate_counts["valid_vote_values"].ne(1)
    | candidate_counts["rejected_values"].ne(1)
    | candidate_counts["source_values"].ne(1)
].copy()

print("\nCOUNTIES WITH STRUCTURAL INCONSISTENCIES")
display(inconsistent_counties)

# ------------------------------------------------
# STEP 4: COMPARE CANDIDATE SUMS WITH VALID VOTES
# ------------------------------------------------

county_vote_checks = (
    presidential_2022
    .groupby("County", dropna=False)
    .agg(
        candidate_vote_sum=("Votes", "sum"),
        reported_valid_votes=("Valid votes", "first"),
        registered_voters=("Registered voters", "first"),
        rejected_ballots=("Rejected ballots", "first"),
        source=("Source", "first")
    )
    .reset_index()
)

county_vote_checks["candidate_valid_difference"] = (
    county_vote_checks["candidate_vote_sum"]
    - county_vote_checks["reported_valid_votes"]
)

vote_exceptions = county_vote_checks.loc[
    county_vote_checks["candidate_valid_difference"].ne(0)
].copy()

print("\nCANDIDATE SUM VS REPORTED VALID VOTES")
print("County records:", len(county_vote_checks))
print("Counties with differences:", len(vote_exceptions))

display(vote_exceptions)

# ------------------------------------------------
# STEP 5: INSPECT SOURCE PROVENANCE
# ------------------------------------------------

print("\nPRESIDENTIAL SOURCE DISTRIBUTION")
display(
    presidential_2022["Source"]
    .value_counts(dropna=False)
    .to_frame("records")
)

# ------------------------------------------------
# STEP 6: INSPECT SOURCE NOTES
# ------------------------------------------------

source_notes = presidential_2022.loc[
    presidential_2022["Note"].notna(),
    ["County", "Candidate", "Note"]
].drop_duplicates()

print("\nSOURCE NOTES")
display(source_notes)

# ------------------------------------------------
# STEP 7: SUMMARY
# ------------------------------------------------

print("\nCELL 04 — VALIDATION SUMMARY")
print("=" * 65)

print("Presidential rows:", len(presidential_2022))
print("County labels:", presidential_2022["County"].nunique())
print("Structural exceptions:", len(inconsistent_counties))
print("Candidate/valid-vote exceptions:", len(vote_exceptions))

print("\nCELL 04 INSPECTION: COMPLETE")


2022 PRESIDENTIAL RECORDS
Presidential rows: 196
Distinct county labels: 49

COUNTY LABELS
['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Citizens residing outside Kenya', 'Elgeyo/Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado', 'Kakamega', 'Kenya', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisii', 'Kisumu', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Machakos', 'Makueni', 'Mandera', 'Marsabit', 'Meru', 'Migori', 'Mombasa', "Murang'a", 'Nairobi City', 'Nakuru', 'Nandi', 'Narok', 'Nyamira', 'Nyandarua', 'Nyeri', 'Samburu', 'Siaya', 'Taita Taveta', 'Tana River', 'Tharaka - Nithi', 'Trans Nzoia', 'Turkana', 'Uasin Gishu', 'Vihiga', 'Wajir', 'West Pokot']

CANDIDATE AND REPEATED-TOTAL CHECKS


,County,candidate_rows,unique_candidates,registered_values,valid_vote_values,rejected_values,source_values
0,Baringo,4,4,1,1,1,1
1,Bomet,4,4,1,1,1,1
2,Bungoma,4,4,1,1,1,1
3,Busia,4,4,1,1,1,1
4,Citizens residing outside Kenya,4,4,1,1,1,1
5,Elgeyo/Marakwet,4,4,1,1,1,1
6,Embu,4,4,1,1,1,1
7,Garissa,4,4,1,1,1,1
8,Homa Bay,4,4,1,1,1,1
9,Isiolo,4,4,1,1,1,1



COUNTIES WITH STRUCTURAL INCONSISTENCIES


,County,candidate_rows,unique_candidates,registered_values,valid_vote_values,rejected_values,source_values



CANDIDATE SUM VS REPORTED VALID VOTES
County records: 49
Counties with differences: 3


,County,candidate_vote_sum,reported_valid_votes,registered_voters,rejected_ballots,source,candidate_valid_difference
12,Kenya,14213027.0,14213137.0,22120458.0,113614.0,"Form 34C, 15 August 2022; Gazette Notice No. 9773",-110.0
15,Kilifi,285486.0,285496.0,588842.0,3191.0,"Form 34C, 15 August 2022",-10.0
48,West Pokot,173605.0,173705.0,220042.0,1258.0,"Form 34C, 15 August 2022",-100.0



PRESIDENTIAL SOURCE DISTRIBUTION


,records
Source,
"Form 34C, 15 August 2022",192
"Form 34C, 15 August 2022; Gazette Notice No. 9773",4



SOURCE NOTES


,County,Candidate,Note
8,Kilifi,Odinga Raila,"On the form the four candidates add up to 285,..."
9,Kilifi,Ruto William Samoei,"On the form the four candidates add up to 285,..."
10,Kilifi,Waihiga David Mwaure,"On the form the four candidates add up to 285,..."
11,Kilifi,Wajackoyah George Luchiri,"On the form the four candidates add up to 285,..."
92,West Pokot,Odinga Raila,"On the form the four candidates add up to 173,..."
93,West Pokot,Ruto William Samoei,"On the form the four candidates add up to 173,..."
94,West Pokot,Waihiga David Mwaure,"On the form the four candidates add up to 173,..."
95,West Pokot,Wajackoyah George Luchiri,"On the form the four candidates add up to 173,..."
192,Kenya,Odinga Raila,"On the form the four candidates add up to 14,2..."
193,Kenya,Ruto William Samoei,"On the form the four candidates add up to 14,2..."



CELL 04 — VALIDATION SUMMARY
Presidential rows: 196
County labels: 49
Structural exceptions: 0
Candidate/valid-vote exceptions: 3

CELL 04 INSPECTION: COMPLETE


The 49 groups consist of Kenya's 47 counties, one diaspora group and one national summary. We must exclude the latter two from the county-level reconciliation, while retaining them for separate aggregate checks.

### Discrepancies identified
| Geography	| Candidate-vote | sum	| Reported valid votes	| Difference| 
|--|--|--|--|--|
| Kilifi	| 285,486	| 285,496	| −10|
| West Pokot	| 173,605	| 173,705	| −100|
| Kenya (national summary)	| 14,213,027	| 14,213,137	| −110|

The two county differences add up to the 110-vote national discrepancy.

This is an important observation: the national difference is arithmetically consistent with the two county-level differences. It does not establish which figures are correct.

We should preserve both the candidate sums and the reported valid-vote totals. We should also retain Takwimu's explanatory notes.

### What this means for our project
The Takwimu dataset is suitable for a preliminary comparison against ELOG because it provides registered voters and reported valid votes for each county.

However, the source is still a transcription attributed to IEBC Form 34C. We have not yet independently authenticated the original declaration.

Therefore, our next objective is to answer three questions:
- Do ELOG and Takwimu report the same registered-voter figures for all 47 counties?
- Do their reported valid-vote totals agree?
- Are any discrepancies concentrated in the counties already flagged by Takwimu?
We will not change the analytical dataset until we have answered these questions.

### Build the 2022 county reconciliation
This cell creates a county-level comparison using the existing ELOG baseline and Takwimu's reported county totals.

It will also preserve the candidate-vote sums and source notes so that differences remain traceable.

In [18]:

# ================================================================
# NOTEBOOK 06 — CELL 05
# RECONCILE 2022 ELOG AND TAKWIMU COUNTY FIGURES
# ================================================================
# Purpose:
# 1. Exclude diaspora and national summary records.
# 2. Create one Takwimu record per county.
# 3. Preserve candidate-vote discrepancies and source notes.
# 4. Compare registration and valid-vote figures with ELOG.
# 5. Identify unmatched counties and numerical differences.
#
# Read-only analysis: no source or analytical files modified.
# ================================================================

import pandas as pd
import numpy as np
import re
import unicodedata

# ------------------------------------------------
# STEP 1: NORMALISE COUNTY KEYS
# ------------------------------------------------

def normalise_county_key(value):
    """Standardise county labels for source matching."""
    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize("NFKD", str(value))
    value = value.upper()

    return re.sub(r"[^A-Z0-9]", "", value)


# ------------------------------------------------
# STEP 2: REMOVE NON-COUNTY SUMMARY RECORDS
# ------------------------------------------------

excluded_geographies = [
    "Kenya",
    "Citizens residing outside Kenya"
]

takwimu_county_rows = presidential_2022.loc[
    ~presidential_2022["County"].isin(
        excluded_geographies
    )
].copy()

takwimu_county_rows["county_key"] = (
    takwimu_county_rows["County"]
    .apply(normalise_county_key)
)

# Confirm county coverage before aggregation.
assert len(takwimu_county_rows) == 188
assert takwimu_county_rows["county_key"].nunique() == 47

# ------------------------------------------------
# STEP 3: PREPARE COUNTY-LEVEL TAKWIMU FIGURES
# ------------------------------------------------

def combine_unique_notes(series):
    """Preserve unique non-empty source notes."""
    notes = [
        str(value).strip()
        for value in series.dropna().unique()
        if str(value).strip()
    ]
    return " | ".join(notes)


takwimu_county_2022 = (
    takwimu_county_rows
    .groupby("county_key", as_index=False)
    .agg(
        takwimu_county=("County", "first"),
        takwimu_registered=("Registered voters", "first"),
        takwimu_valid_votes=("Valid votes", "first"),
        takwimu_rejected_ballots=("Rejected ballots", "first"),
        takwimu_candidate_vote_sum=("Votes", "sum"),
        takwimu_source=("Source", "first"),
        takwimu_notes=("Note", combine_unique_notes)
    )
)

takwimu_county_2022["candidate_valid_difference"] = (
    takwimu_county_2022["takwimu_candidate_vote_sum"]
    - takwimu_county_2022["takwimu_valid_votes"]
)

# ------------------------------------------------
# STEP 4: PREPARE EXISTING ELOG BASELINE
# ------------------------------------------------

elog_county_2022 = baseline_2022[
    [
        "county",
        "county_key",
        "analytical_registered_voters",
        "analytical_valid_votes",
        "validation_status"
    ]
].copy()

elog_county_2022 = elog_county_2022.rename(
    columns={
        "county": "elog_county",
        "analytical_registered_voters": "elog_registered",
        "analytical_valid_votes": "elog_valid_votes"
    }
)

elog_county_2022["county_key"] = (
    elog_county_2022["county_key"]
    .apply(normalise_county_key)
)

assert elog_county_2022["county_key"].nunique() == 47

# ------------------------------------------------
# STEP 5: RECONCILE BOTH SOURCES
# ------------------------------------------------

reconciliation_2022 = elog_county_2022.merge(
    takwimu_county_2022,
    on="county_key",
    how="outer",
    validate="one_to_one",
    indicator=True
)

reconciliation_2022["registered_difference"] = (
    reconciliation_2022["elog_registered"]
    - reconciliation_2022["takwimu_registered"]
)

reconciliation_2022["valid_vote_difference"] = (
    reconciliation_2022["elog_valid_votes"]
    - reconciliation_2022["takwimu_valid_votes"]
)

reconciliation_2022["registration_match"] = (
    reconciliation_2022["registered_difference"].eq(0)
)

reconciliation_2022["valid_vote_match"] = (
    reconciliation_2022["valid_vote_difference"].eq(0)
)

# ------------------------------------------------
# STEP 6: REPORT COUNTY COVERAGE
# ------------------------------------------------

print("2022 ELOG–TAKWIMU COUNTY RECONCILIATION")
print("=" * 65)

print("ELOG counties:", len(elog_county_2022))
print("Takwimu counties:", len(takwimu_county_2022))

print("\nMERGE COVERAGE")
print(
    reconciliation_2022["_merge"]
    .value_counts(dropna=False)
    .to_string()
)

unmatched_counties = reconciliation_2022.loc[
    reconciliation_2022["_merge"].ne("both")
]

print("\nUNMATCHED COUNTIES")
display(unmatched_counties)

# ------------------------------------------------
# STEP 7: REPORT NUMERICAL AGREEMENT
# ------------------------------------------------

matched_counties = reconciliation_2022.loc[
    reconciliation_2022["_merge"].eq("both")
].copy()

print("\nMATCHED COUNTY CHECKS")
print("Matched counties:", len(matched_counties))

print(
    "Registration matches:",
    int(matched_counties["registration_match"].sum())
)

print(
    "Valid-vote matches:",
    int(matched_counties["valid_vote_match"].sum())
)

registration_exceptions = matched_counties.loc[
    ~matched_counties["registration_match"]
]

valid_vote_exceptions = matched_counties.loc[
    ~matched_counties["valid_vote_match"]
]

print("\nREGISTRATION EXCEPTIONS")
display(
    registration_exceptions[
        [
            "elog_county",
            "elog_registered",
            "takwimu_registered",
            "registered_difference"
        ]
    ]
)

print("\nVALID-VOTE EXCEPTIONS")
display(
    valid_vote_exceptions[
        [
            "elog_county",
            "elog_valid_votes",
            "takwimu_valid_votes",
            "valid_vote_difference",
            "takwimu_candidate_vote_sum",
            "candidate_valid_difference",
            "takwimu_notes"
        ]
    ]
)

# ------------------------------------------------
# STEP 8: COMPARE SOURCE TOTALS
# ------------------------------------------------

print("\nSOURCE TOTALS — 47 COUNTIES")
print("=" * 65)

totals_comparison = pd.DataFrame({
    "measure": [
        "Registered voters",
        "Valid votes"
    ],
    "ELOG": [
        elog_county_2022["elog_registered"].sum(),
        elog_county_2022["elog_valid_votes"].sum()
    ],
    "Takwimu": [
        takwimu_county_2022["takwimu_registered"].sum(),
        takwimu_county_2022["takwimu_valid_votes"].sum()
    ]
})

totals_comparison["ELOG_minus_Takwimu"] = (
    totals_comparison["ELOG"]
    - totals_comparison["Takwimu"]
)

display(totals_comparison)

print("\nCELL 05 RECONCILIATION: COMPLETE")


2022 ELOG–TAKWIMU COUNTY RECONCILIATION
ELOG counties: 47
Takwimu counties: 47

MERGE COVERAGE
_merge
both          47
left_only      0
right_only     0

UNMATCHED COUNTIES


,elog_county,county_key,elog_registered,elog_valid_votes,validation_status,takwimu_county,takwimu_registered,takwimu_valid_votes,takwimu_rejected_ballots,takwimu_candidate_vote_sum,takwimu_source,takwimu_notes,candidate_valid_difference,_merge,registered_difference,valid_vote_difference,registration_match,valid_vote_match



MATCHED COUNTY CHECKS
Matched counties: 47
Registration matches: 47
Valid-vote matches: 47

REGISTRATION EXCEPTIONS


,elog_county,elog_registered,takwimu_registered,registered_difference



VALID-VOTE EXCEPTIONS


,elog_county,elog_valid_votes,takwimu_valid_votes,valid_vote_difference,takwimu_candidate_vote_sum,candidate_valid_difference,takwimu_notes



SOURCE TOTALS — 47 COUNTIES


,measure,ELOG,Takwimu,ELOG_minus_Takwimu
0,Registered voters,22110015.0,22110015.0,0.0
1,Valid votes,14207149.0,14207149.0,0.0



CELL 05 RECONCILIATION: COMPLETE


We now have evidence that the ELOG-derived 2022 county figures agree exactly with a second published dataset attributed to IEBC Form 34C.

This substantially strengthens confidence in the figures used for our 2022 analysis.

However, we need to distinguish agreement between published datasets from verification against the original signed declaration. The latter remains outstanding.

There is also an important caveat: we have not established whether ELOG and Takwimu independently transcribed their figures or share an underlying source. Their agreement is strong evidence of consistency, but not proof of independent accuracy.

### What happens to the Kilifi and West Pokot discrepancies?
We retain them in our documentation.

Recall that Takwimu reported these internal differences:
| County	| Reported valid votes	| Candidate-vote sum	| Difference|
|--|--|--|--|
| Kilifi	| 285,496	| 285,486	| −10|
| West Pokot	| 173,705	| 173,605	| −100|

ELOG agrees with Takwimu's reported valid-vote totals, not necessarily the sums of the four candidate records.

Therefore, our reconciliation establishes agreement between the reported county totals. It does not resolve the underlying candidate-sum discrepancies.

We should not change either county's figures without examining the original declaration.

### Save the reconciliation evidence
Before proceeding, we should preserve the 47-county reconciliation and create a validation summary.

This cell will save three files under documentation/ and data/Interim/. It will not modify the existing analytical dataset.

In [23]:

# ================================================================
# NOTEBOOK 06 — CELL 06
# SAVE 2022 SECONDARY-SOURCE RECONCILIATION
# ================================================================
# Purpose:
# 1. Preserve the 47-county reconciliation.
# 2. Document candidate-sum discrepancies.
# 3. Save a source-verification summary.
# 4. Keep original ELOG and analytical files unchanged.
# ================================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------
# STEP 1: DEFINE OUTPUT FILES
# ------------------------------------------------

reconciliation_path = (
    DOCS_DIR / "reconciliation_2022_elog_takwimu.csv"
)

exceptions_path = (
    DOCS_DIR / "validation_register_2022.csv"
)

summary_path = (
    DOCS_DIR / "notebook06_2022_source_validation_summary.csv"
)

takwimu_clean_path = (
    INTERIM_DIR / "takwimu_2022_county_presidential.csv"
)

output_paths = [
    reconciliation_path,
    exceptions_path,
    summary_path,
    takwimu_clean_path
]

# Protect existing files.
for path in output_paths:
    assert not path.exists(), (
        f"File already exists: {path}"
    )

# ------------------------------------------------
# STEP 2: BUILD THE VALIDATION REGISTER
# ------------------------------------------------

validation_register_2022 = reconciliation_2022.copy()

validation_register_2022["source_comparison_status"] = (
    "ELOG_TAKWIMU_MATCH"
)

validation_register_2022.loc[
    ~validation_register_2022["registration_match"]
    | ~validation_register_2022["valid_vote_match"],
    "source_comparison_status"
] = "SOURCE_DIFFERENCE"

validation_register_2022.loc[
    validation_register_2022["_merge"].ne("both"),
    "source_comparison_status"
] = "COUNTY_UNMATCHED"

validation_register_2022["candidate_sum_exception"] = (
    validation_register_2022[
        "candidate_valid_difference"
    ].ne(0)
)

validation_register_2022["official_document_verified"] = False

validation_register_2022["verification_level"] = (
    "SECONDARY_SOURCE_RECONCILED"
)

# ------------------------------------------------
# STEP 3: BUILD SUMMARY
# ------------------------------------------------

validation_summary_2022 = pd.DataFrame([
    {
        "check": "Matched counties",
        "actual": int(
            reconciliation_2022["_merge"].eq("both").sum()
        ),
        "expected": 47
    },
    {
        "check": "Registration matches",
        "actual": int(
            reconciliation_2022["registration_match"].sum()
        ),
        "expected": 47
    },
    {
        "check": "Valid-vote matches",
        "actual": int(
            reconciliation_2022["valid_vote_match"].sum()
        ),
        "expected": 47
    },
    {
        "check": "Candidate-sum exceptions",
        "actual": int(
            validation_register_2022[
                "candidate_sum_exception"
            ].sum()
        ),
        "expected": 2
    }
])

validation_summary_2022["status"] = (
    validation_summary_2022.apply(
        lambda row: (
            "PASS"
            if row["actual"] == row["expected"]
            else "REVIEW"
        ),
        axis=1
    )
)

# ------------------------------------------------
# STEP 4: VALIDATE BEFORE EXPORT
# ------------------------------------------------

assert len(reconciliation_2022) == 47
assert reconciliation_2022["_merge"].eq("both").all()

assert reconciliation_2022["registration_match"].all()
assert reconciliation_2022["valid_vote_match"].all()

assert (
    validation_register_2022[
        "candidate_sum_exception"
    ].sum() == 2
)

assert validation_summary_2022["status"].eq("PASS").all()

# ------------------------------------------------
# STEP 5: EXPORT DOCUMENTATION
# ------------------------------------------------

reconciliation_2022.to_csv(
    reconciliation_path,
    index=False
)

validation_register_2022.to_csv(
    exceptions_path,
    index=False
)

validation_summary_2022.to_csv(
    summary_path,
    index=False
)

takwimu_county_2022.to_csv(
    takwimu_clean_path,
    index=False
)

# ------------------------------------------------
# STEP 6: VERIFY EXPORTED FILES
# ------------------------------------------------

print("NOTEBOOK 06 — 2022 RECONCILIATION EXPORT")
print("=" * 65)

for path in output_paths:
    assert path.is_file()
    print("Saved:", path)

print("\nVALIDATION SUMMARY")
display(validation_summary_2022)

print("\nSOURCE VERIFICATION STATUS")
print("County-level ELOG–Takwimu matches: 47/47")
print("Candidate-sum exceptions retained: 2")
print("Official Form 34C authenticated: No")

print("\nCELL 06 EXPORT VERIFICATION: PASSED")


NOTEBOOK 06 — 2022 RECONCILIATION EXPORT
Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\reconciliation_2022_elog_takwimu.csv
Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\validation_register_2022.csv
Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\documentation\notebook06_2022_source_validation_summary.csv
Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\data\Interim\takwimu_2022_county_presidential.csv

VALIDATION SUMMARY


,check,actual,expected,status
0,Matched counties,47,47,PASS
1,Registration matches,47,47,PASS
2,Valid-vote matches,47,47,PASS
3,Candidate-sum exceptions,2,2,PASS



SOURCE VERIFICATION STATUS
County-level ELOG–Takwimu matches: 47/47
Candidate-sum exceptions retained: 2
Official Form 34C authenticated: No

CELL 06 EXPORT VERIFICATION: PASSED


### The four output files have been saved successfully, and all four validation checks passed.
**What we can now conclude**

The ELOG and Takwimu datasets agree exactly on the registered-voter and reported valid-vote figures for all 47 counties in 2022.

However, we must preserve two qualifications:
- Kilifi: The four candidate votes sum to 10 fewer than the reported valid-vote total.
- West Pokot: The four candidate votes sum to 100 fewer than the reported valid-vote total.
These discrepancies do not create ELOG–Takwimu differences because both sources agree on the reported county totals. They remain internal source-accounting exceptions.

Also, the original signed IEBC Form 34C has not yet been authenticated. Our verification level is therefore secondary-source reconciled, not officially verified.

### Final source-assurance review
Before drafting the final portfolio report, I recommend consolidating the verification status across all four electoral events.

This will give us a clear, auditable answer to the question raised when we started Notebook 06: How reliable are our figures, and how do we explain the source limitations?
Notebook 06 — Cell 07
This cell reads the saved validation summaries, checks that the 2022 evidence was exported correctly, and creates a final source-assurance overview.

It will not alter the analytical dataset.

In [27]:

# ================================================================
# NOTEBOOK 06 — CELL 07
# FINAL SOURCE-ASSURANCE REVIEW
# ================================================================
# Purpose:
# 1. Verify the saved 2022 reconciliation.
# 2. Review existing analytical validation statuses.
# 3. Summarise evidence across all four elections.
# 4. Preserve distinctions between verification levels.
#
# Read-only: no files are created or modified.
# ================================================================

import pandas as pd

# ------------------------------------------------
# STEP 1: LOAD SAVED 2022 RECONCILIATION
# ------------------------------------------------

reconciliation_path = (
    DOCS_DIR / "reconciliation_2022_elog_takwimu.csv"
)

validation_path = (
    DOCS_DIR / "validation_register_2022.csv"
)

summary_path = (
    DOCS_DIR / "notebook06_2022_source_validation_summary.csv"
)

saved_reconciliation = pd.read_csv(reconciliation_path)
saved_validation = pd.read_csv(validation_path)
saved_summary = pd.read_csv(summary_path)

assert len(saved_reconciliation) == 47
assert len(saved_validation) == 47
assert saved_summary["status"].eq("PASS").all()

# ------------------------------------------------
# STEP 2: REVIEW ANALYTICAL DATASET STATUSES
# ------------------------------------------------

source_status_summary = (
    elections_df
    .groupby(
        ["election_id", "validation_status"],
        dropna=False
    )
    .agg(
        counties=("county_key", "nunique"),
        records=("county_key", "size")
    )
    .reset_index()
)

print("ANALYTICAL DATASET — EXISTING VALIDATION STATUSES")
print("=" * 65)

display(source_status_summary)

# ------------------------------------------------
# STEP 3: BUILD SOURCE-ASSURANCE OVERVIEW
# ------------------------------------------------

source_assurance = pd.DataFrame([
    {
        "election_id": "2013_GENERAL",
        "primary_source": "ELOG",
        "comparison_source": "IEBC",
        "county_coverage": 47,
        "source_assurance_level": "PARTIALLY_RECONCILED",
        "outstanding_limitation": (
            "Documented registration difference "
            "against national benchmark"
        )
    },
    {
        "election_id": "2017_AUG_GENERAL",
        "primary_source": "ELOG",
        "comparison_source": "IEBC",
        "county_coverage": 47,
        "source_assurance_level": "EXCEPTIONS_DOCUMENTED",
        "outstanding_limitation": (
            "Kisumu and Turkana valid-vote "
            "discrepancies unresolved"
        )
    },
    {
        "election_id": "2017_OCT_FRESH",
        "primary_source": "ELOG",
        "comparison_source": "IEBC",
        "county_coverage": 47,
        "source_assurance_level": "COUNTY_RECONCILED",
        "outstanding_limitation": (
            "Fresh presidential election; "
            "not an ordinary general election"
        )
    },
    {
        "election_id": "2022_GENERAL",
        "primary_source": "ELOG",
        "comparison_source": "Takwimu transcription",
        "county_coverage": 47,
        "source_assurance_level": "SECONDARY_SOURCE_RECONCILED",
        "outstanding_limitation": (
            "Original Form 34C not authenticated; "
            "Kilifi and West Pokot candidate-sum "
            "exceptions retained"
        )
    }
])

print("\nFINAL SOURCE-ASSURANCE OVERVIEW")
print("=" * 65)

display(source_assurance)

# ------------------------------------------------
# STEP 4: VERIFY 2022 ASSURANCE CLAIMS
# ------------------------------------------------

assert saved_reconciliation["_merge"].eq("both").all()
assert saved_reconciliation["registration_match"].all()
assert saved_reconciliation["valid_vote_match"].all()

assert (
    saved_validation["candidate_sum_exception"].sum() == 2
)

assert (
    saved_validation["official_document_verified"].eq(False).all()
)

print("\n2022 ASSURANCE CHECKS")
print("Matched counties: 47")
print("Registration matches: 47")
print("Valid-vote matches: 47")
print("Candidate-sum exceptions: 2")
print("Original Form 34C authenticated: No")

print("\nCELL 07 SOURCE-ASSURANCE REVIEW: PASSED")


ANALYTICAL DATASET — EXISTING VALIDATION STATUSES


,election_id,validation_status,counties,records
0,2013_GENERAL,PARTIALLY_RECONCILED,47,47
1,2017_AUG_GENERAL,EXCEPTIONS_DOCUMENTED,47,47
2,2017_OCT_FRESH,COUNTY_RECONCILED,47,47
3,2022_GENERAL,COUNTY_SOURCE_UNVERIFIED,47,47



FINAL SOURCE-ASSURANCE OVERVIEW


,election_id,primary_source,comparison_source,county_coverage,source_assurance_level,outstanding_limitation
0,2013_GENERAL,ELOG,IEBC,47,PARTIALLY_RECONCILED,Documented registration difference against nat...
1,2017_AUG_GENERAL,ELOG,IEBC,47,EXCEPTIONS_DOCUMENTED,Kisumu and Turkana valid-vote discrepancies un...
2,2017_OCT_FRESH,ELOG,IEBC,47,COUNTY_RECONCILED,Fresh presidential election; not an ordinary g...
3,2022_GENERAL,ELOG,Takwimu transcription,47,SECONDARY_SOURCE_RECONCILED,Original Form 34C not authenticated; Kilifi an...



2022 ASSURANCE CHECKS
Matched counties: 47
Registration matches: 47
Valid-vote matches: 47
Candidate-sum exceptions: 2
Original Form 34C authenticated: No

CELL 07 SOURCE-ASSURANCE REVIEW: PASSED


### Consolidate final analytical findings
We will use the saved findings from Notebook 04 and the reconciliation evidence from Notebook 06.

This cell inspects the findings structure before we construct the final report. It will not modify any files.

In [30]:

# ================================================================
# NOTEBOOK 06 — CELL 08
# INSPECT FINAL ANALYTICAL FINDINGS
# ================================================================
# Purpose:
# 1. Load the saved Notebook 04 key findings.
# 2. Inspect the structure and available metrics.
# 3. Confirm the final source-assurance overview.
# 4. Prepare for automated portfolio reporting.
#
# Read-only: no files are modified.
# ================================================================

import pandas as pd

# ------------------------------------------------
# STEP 1: LOCATE SAVED FINDINGS
# ------------------------------------------------

findings_path = (
    DOCS_DIR / "notebook04_key_findings.csv"
)

assert findings_path.is_file(), (
    f"Findings file not found: {findings_path}"
)

# ------------------------------------------------
# STEP 2: LOAD THE FINDINGS
# ------------------------------------------------

key_findings = pd.read_csv(findings_path)

print("NOTEBOOK 06 — FINAL FINDINGS INSPECTION")
print("=" * 65)

print("Findings records:", len(key_findings))
print("Columns:", len(key_findings.columns))

print("\nCOLUMN NAMES")
print(key_findings.columns.tolist())

print("\nSAVED KEY FINDINGS")
display(key_findings)

# ------------------------------------------------
# STEP 3: VERIFY SOURCE ASSURANCE
# ------------------------------------------------

assert len(source_assurance) == 4
assert source_assurance["election_id"].nunique() == 4

assert (
    source_assurance.loc[
        source_assurance["election_id"].eq(
            "2022_GENERAL"
        ),
        "source_assurance_level"
    ].iloc[0]
    == "SECONDARY_SOURCE_RECONCILED"
)

print("\nSOURCE-ASSURANCE OVERVIEW")
display(source_assurance)

# ------------------------------------------------
# STEP 4: CHECK REPORTING ASSETS
# ------------------------------------------------

report_path = (
    REPORTS_DIR / "notebook05_findings_report.md"
)

inventory_path = (
    DOCS_DIR / "notebook05_chart_inventory.csv"
)

assert report_path.is_file()
assert inventory_path.is_file()

chart_inventory = pd.read_csv(inventory_path)

assert len(chart_inventory) == 5
assert chart_inventory["status"].eq("VERIFIED").all()

print("\nREPORTING ASSETS")
print("Notebook 05 findings report: Available")
print("Verified charts:", len(chart_inventory))
print("Notebook 04 key findings: Available")

print("\nCELL 08 FINDINGS INSPECTION: PASSED")


NOTEBOOK 06 — FINAL FINDINGS INSPECTION
Findings records: 10
Columns: 3

COLUMN NAMES
['finding', 'value', 'unit']

SAVED KEY FINDINGS


,finding,value,unit
0,Comparable counties,45.000,counties
1,Counties with declining valid-vote participation,45.000,counties
2,"Mean county participation decline, 2013-2022",19.360,percentage points
3,"Aggregate registration growth, 2013-2022",53.360,percent
4,"Aggregate valid-vote growth, 2013-2022",15.820,percent
5,Counties with increased absolute valid votes,42.000,counties
6,Counties with decreased absolute valid votes,3.000,counties
7,Counties with declining August-October 2017 tu...,45.000,counties
8,Pearson correlation: registration growth vs pa...,0.297,correlation coefficient
9,Spearman correlation: registration growth vs p...,0.319,correlation coefficient



SOURCE-ASSURANCE OVERVIEW


,election_id,primary_source,comparison_source,county_coverage,source_assurance_level,outstanding_limitation
0,2013_GENERAL,ELOG,IEBC,47,PARTIALLY_RECONCILED,Documented registration difference against nat...
1,2017_AUG_GENERAL,ELOG,IEBC,47,EXCEPTIONS_DOCUMENTED,Kisumu and Turkana valid-vote discrepancies un...
2,2017_OCT_FRESH,ELOG,IEBC,47,COUNTY_RECONCILED,Fresh presidential election; not an ordinary g...
3,2022_GENERAL,ELOG,Takwimu transcription,47,SECONDARY_SOURCE_RECONCILED,Original Form 34C not authenticated; Kilifi an...



REPORTING ASSETS
Notebook 05 findings report: Available
Verified charts: 5
Notebook 04 key findings: Available

CELL 08 FINDINGS INSPECTION: PASSED


### One methodological distinction to preserve
We should be particularly careful with the wording of the final report.

Valid-vote participation is not the same as voter turnout.

Our principal 2013–2022 comparison measures:
$$
\text{Valid-vote participation (\%)} =
\frac{\text{Valid votes}}{\text{Registered voters}}\times100
$$

For the August–October 2017 comparison, we use turnout based on ballots cast, where reconciled figures are available.

We also need to state clearly that the main longitudinal analysis covers 45 comparable counties, excluding Kisumu and Turkana because of unresolved August 2017 reconciliation exceptions.

### Final synthesis consistency checks

In [34]:

# ================================================================
# NOTEBOOK 06 — CELL 09
# FINAL SYNTHESIS CONSISTENCY CHECKS
# ================================================================
# Purpose:
# 1. Recompute key findings from the analytical dataset.
# 2. Compare recomputed figures with Notebook 04 findings.
# 3. Verify source-reconciliation evidence.
# 4. Confirm the five final chart files exist.
#
# Read-only: no files are created or modified.
# ================================================================

import numpy as np
import pandas as pd

# ------------------------------------------------
# STEP 1: DEFINE COMPARABLE GENERAL-ELECTION SAMPLE
# ------------------------------------------------

general_ids = [
    "2013_GENERAL",
    "2017_AUG_GENERAL",
    "2022_GENERAL"
]

general_df = elections_df.loc[
    elections_df["election_id"].isin(general_ids)
].copy()

general_df["analytical_registered_voters"] = (
    pd.to_numeric(
        general_df["analytical_registered_voters"],
        errors="coerce"
    )
)

general_df["analytical_valid_votes"] = (
    pd.to_numeric(
        general_df["analytical_valid_votes"],
        errors="coerce"
    )
)

# Require complete valid-vote participation across all
# three general elections.
participation_wide = general_df.pivot(
    index="county_key",
    columns="election_id",
    values="valid_vote_participation_pct"
)

comparable_keys = participation_wide.dropna(
    subset=general_ids
).index

assert len(comparable_keys) == 45, (
    "Expected 45 comparable counties."
)

comparable_general = general_df.loc[
    general_df["county_key"].isin(comparable_keys)
].copy()

# ------------------------------------------------
# STEP 2: RECOMPUTE PARTICIPATION CHANGES
# ------------------------------------------------

comparable_participation = participation_wide.loc[
    comparable_keys
].copy()

participation_change = (
    comparable_participation["2022_GENERAL"]
    - comparable_participation["2013_GENERAL"]
)

declining_counties = int(
    participation_change.lt(0).sum()
)

mean_decline_pp = float(
    -participation_change.mean()
)

# ------------------------------------------------
# STEP 3: RECOMPUTE AGGREGATE GROWTH
# ------------------------------------------------

aggregate_totals = (
    comparable_general
    .groupby("election_id")
    .agg(
        registered=("analytical_registered_voters", "sum"),
        valid_votes=("analytical_valid_votes", "sum")
    )
)

registration_growth_pct = (
    (
        aggregate_totals.loc[
            "2022_GENERAL", "registered"
        ]
        / aggregate_totals.loc[
            "2013_GENERAL", "registered"
        ]
    ) - 1
) * 100

valid_vote_growth_pct = (
    (
        aggregate_totals.loc[
            "2022_GENERAL", "valid_votes"
        ]
        / aggregate_totals.loc[
            "2013_GENERAL", "valid_votes"
        ]
    ) - 1
) * 100

# ------------------------------------------------
# STEP 4: COMPARE WITH SAVED KEY FINDINGS
# ------------------------------------------------

saved_findings = (
    key_findings
    .set_index("finding")["value"]
)

checks = [
    {
        "check": "Comparable counties",
        "recomputed": len(comparable_keys),
        "saved": saved_findings[
            "Comparable counties"
        ],
        "tolerance": 0
    },
    {
        "check": "Counties with declining participation",
        "recomputed": declining_counties,
        "saved": saved_findings[
            "Counties with declining valid-vote participation"
        ],
        "tolerance": 0
    },
    {
        "check": "Mean participation decline",
        "recomputed": mean_decline_pp,
        "saved": saved_findings[
            "Mean county participation decline, 2013-2022"
        ],
        "tolerance": 0.02
    },
    {
        "check": "Aggregate registration growth",
        "recomputed": registration_growth_pct,
        "saved": saved_findings[
            "Aggregate registration growth, 2013-2022"
        ],
        "tolerance": 0.02
    },
    {
        "check": "Aggregate valid-vote growth",
        "recomputed": valid_vote_growth_pct,
        "saved": saved_findings[
            "Aggregate valid-vote growth, 2013-2022"
        ],
        "tolerance": 0.02
    }
]

consistency_checks = pd.DataFrame(checks)

consistency_checks["difference"] = (
    consistency_checks["recomputed"]
    - consistency_checks["saved"]
).abs()

consistency_checks["status"] = np.where(
    consistency_checks["difference"]
    <= consistency_checks["tolerance"],
    "PASS",
    "REVIEW"
)

print("FINAL SYNTHESIS — METRIC CONSISTENCY")
print("=" * 65)
display(consistency_checks)

# ------------------------------------------------
# STEP 5: VERIFY CHART FILES
# ------------------------------------------------

expected_charts = [
    "01_general_election_valid_vote_participation.png",
    "02_county_participation_declines_2013_2022.png",
    "03_registration_vs_valid_vote_growth_2013_2022.png",
    "04_august_october_2017_county_turnout_comparison.png",
    "05_august_october_2017_turnout_decline_distribution.png"
]

chart_checks = pd.DataFrame([
    {
        "chart": chart,
        "exists": (FIGURES_DIR / chart).is_file()
    }
    for chart in expected_charts
])

print("\nFINAL CHART FILE VERIFICATION")
display(chart_checks)

# ------------------------------------------------
# STEP 6: VERIFY 2022 SOURCE EVIDENCE
# ------------------------------------------------

assert len(saved_reconciliation) == 47

assert saved_reconciliation["_merge"].eq("both").all()
assert saved_reconciliation["registration_match"].all()
assert saved_reconciliation["valid_vote_match"].all()

assert (
    saved_validation["candidate_sum_exception"].sum() == 2
)

# ------------------------------------------------
# STEP 7: FINAL CHECK
# ------------------------------------------------

assert consistency_checks["status"].eq("PASS").all()
assert chart_checks["exists"].all()

print("\nALL FINAL SYNTHESIS CHECKS: PASSED")


FINAL SYNTHESIS — METRIC CONSISTENCY


,check,recomputed,saved,tolerance,difference,status
0,Comparable counties,45.000000,45.00,0.00,0.000000,PASS
1,Counties with declining participation,45.000000,45.00,0.00,0.000000,PASS
2,Mean participation decline,19.358222,19.36,0.02,0.001778,PASS
3,Aggregate registration growth,53.363096,53.36,0.02,0.003096,PASS
4,Aggregate valid-vote growth,15.817799,15.82,0.02,0.002201,PASS



FINAL CHART FILE VERIFICATION


,chart,exists
0,01_general_election_valid_vote_participation.png,True
1,02_county_participation_declines_2013_2022.png,True
2,03_registration_vs_valid_vote_growth_2013_2022...,True
3,04_august_october_2017_county_turnout_comparis...,True
4,05_august_october_2017_turnout_decline_distrib...,True



ALL FINAL SYNTHESIS CHECKS: PASSED


### Sample coverage sensitivity analysis
This will tell us whether including Kisumu and Turkana changes our central conclusion about declining participation between 2013 and 2022.

The existing 45-county results and charts will remain unchanged.

In [37]:

# ================================================================
# NOTEBOOK 06 — CELL 10
# 45-COUNTY VS 47-COUNTY SENSITIVITY ANALYSIS
# ================================================================
# Purpose:
# 1. Compare the existing 45-county longitudinal sample
#    with all 47 counties for the 2013–2022 endpoints.
# 2. Recalculate participation decline and growth metrics.
# 3. Identify the effect of including Kisumu and Turkana.
#
# Read-only: no files are created or modified.
# ================================================================

import pandas as pd

# ------------------------------------------------
# STEP 1: ISOLATE 2013 AND 2022
# ------------------------------------------------

endpoint_df = elections_df.loc[
    elections_df["election_id"].isin(
        ["2013_GENERAL", "2022_GENERAL"]
    )
].copy()

endpoint_wide = endpoint_df.pivot(
    index="county_key",
    columns="election_id",
    values="valid_vote_participation_pct"
)

endpoint_keys = endpoint_wide.dropna().index

print("2013–2022 ENDPOINT COVERAGE")
print("=" * 65)
print("Counties with both endpoints:", len(endpoint_keys))
print("Existing three-election sample:", len(comparable_keys))

additional_keys = endpoint_keys.difference(comparable_keys)

print("\nCOUNTIES ADDED TO ENDPOINT COMPARISON")
display(
    endpoint_df.loc[
        endpoint_df["county_key"].isin(additional_keys),
        ["county", "election_id", "valid_vote_participation_pct"]
    ].sort_values(["county", "election_id"])
)

# ------------------------------------------------
# STEP 2: CALCULATE METRICS FOR BOTH SAMPLES
# ------------------------------------------------

def calculate_sample_metrics(county_keys, sample_name):

    sample = endpoint_df.loc[
        endpoint_df["county_key"].isin(county_keys)
    ].copy()

    wide = endpoint_wide.loc[county_keys]

    change = (
        wide["2022_GENERAL"]
        - wide["2013_GENERAL"]
    )

    totals = (
        sample.groupby("election_id")
        .agg(
            registered=("analytical_registered_voters", "sum"),
            valid_votes=("analytical_valid_votes", "sum")
        )
    )

    registration_growth = (
        totals.loc["2022_GENERAL", "registered"]
        / totals.loc["2013_GENERAL", "registered"]
        - 1
    ) * 100

    valid_vote_growth = (
        totals.loc["2022_GENERAL", "valid_votes"]
        / totals.loc["2013_GENERAL", "valid_votes"]
        - 1
    ) * 100

    weighted_2013 = (
        totals.loc["2013_GENERAL", "valid_votes"]
        / totals.loc["2013_GENERAL", "registered"]
        * 100
    )

    weighted_2022 = (
        totals.loc["2022_GENERAL", "valid_votes"]
        / totals.loc["2022_GENERAL", "registered"]
        * 100
    )

    return {
        "sample": sample_name,
        "counties": len(county_keys),
        "counties_declining": int(change.lt(0).sum()),
        "mean_decline_pp": -change.mean(),
        "median_decline_pp": -change.median(),
        "weighted_2013_pct": weighted_2013,
        "weighted_2022_pct": weighted_2022,
        "weighted_decline_pp": weighted_2013 - weighted_2022,
        "registration_growth_pct": registration_growth,
        "valid_vote_growth_pct": valid_vote_growth
    }


sensitivity_summary = pd.DataFrame([
    calculate_sample_metrics(
        comparable_keys,
        "45-county three-election sample"
    ),
    calculate_sample_metrics(
        endpoint_keys,
        "47-county endpoint sample"
    )
])

print("\nSAMPLE SENSITIVITY COMPARISON")
display(sensitivity_summary.round(3))

# ------------------------------------------------
# STEP 3: QUANTIFY THE DIFFERENCES
# ------------------------------------------------

numeric_columns = [
    "mean_decline_pp",
    "median_decline_pp",
    "weighted_2013_pct",
    "weighted_2022_pct",
    "weighted_decline_pp",
    "registration_growth_pct",
    "valid_vote_growth_pct"
]

sensitivity_differences = (
    sensitivity_summary.loc[1, numeric_columns]
    - sensitivity_summary.loc[0, numeric_columns]
)

print("\n47-COUNTY MINUS 45-COUNTY DIFFERENCES")
display(
    sensitivity_differences.to_frame(
        "difference"
    ).round(3)
)

print("\nCELL 10 SENSITIVITY ANALYSIS: COMPLETE")


2013–2022 ENDPOINT COVERAGE
Counties with both endpoints: 47
Existing three-election sample: 45

COUNTIES ADDED TO ENDPOINT COMPARISON


,county,election_id,valid_vote_participation_pct
41,Kisumu,2013_GENERAL,89.79
182,Kisumu,2022_GENERAL,70.95
22,Turkana,2013_GENERAL,75.04
163,Turkana,2022_GENERAL,60.17



SAMPLE SENSITIVITY COMPARISON


,sample,counties,counties_declining,mean_decline_pp,median_decline_pp,weighted_2013_pct,weighted_2022_pct,weighted_decline_pp,registration_growth_pct,valid_vote_growth_pct
0,45-county three-election sample,45,45,19.358,20.54,84.895,64.111,20.783,53.363,15.818
1,47-county endpoint sample,47,47,19.252,20.29,84.934,64.257,20.677,53.690,16.273



47-COUNTY MINUS 45-COUNTY DIFFERENCES


,difference
mean_decline_pp,-0.10652
median_decline_pp,-0.25
weighted_2013_pct,0.03939
weighted_2022_pct,0.145287
weighted_decline_pp,-0.105898
registration_growth_pct,0.326559
valid_vote_growth_pct,0.455686



CELL 10 SENSITIVITY ANALYSIS: COMPLETE


### What the sensitivity analysis tells us

The most important result is the change in the weighted participation decline: just 0.106 percentage points.

This suggests that excluding Kisumu and Turkana from the existing 45-county analysis has very little effect on the overall 2013–2022 trend.

However, these differences alone do not confirm whether both additional counties individually recorded declining participation. We would need the full Cell 10 comparison table to establish that.

### One final check before closing Notebook 06
Let's confirm that both Kisumu and Turkana recorded declining participation between 2013 and 2022, and display the complete 47-county results.

**Final sensitivity verification**

In [41]:

# ================================================================
# NOTEBOOK 06 — CELL 11
# FINAL 47-COUNTY ENDPOINT VERIFICATION
# ================================================================
# Purpose:
# 1. Confirm 2013–2022 participation changes for all counties.
# 2. Inspect Kisumu and Turkana individually.
# 3. Display the complete 45-vs-47 county comparison.
# 4. Validate the direction of the national trend.
#
# Read-only: no files modified.
# ================================================================

endpoint_changes = endpoint_wide.loc[
    endpoint_keys
].copy()

endpoint_changes["change_pp"] = (
    endpoint_changes["2022_GENERAL"]
    - endpoint_changes["2013_GENERAL"]
)

# County labels from the analytical dataset
county_lookup = (
    endpoint_df[
        ["county_key", "county"]
    ]
    .drop_duplicates("county_key")
    .set_index("county_key")["county"]
)

endpoint_changes["county"] = (
    endpoint_changes.index.map(county_lookup)
)

print("2013–2022 ENDPOINT PARTICIPATION CHECK")
print("=" * 65)

print("Counties compared:", len(endpoint_changes))
print(
    "Counties with declining participation:",
    int(endpoint_changes["change_pp"].lt(0).sum())
)
print(
    "Counties with increasing participation:",
    int(endpoint_changes["change_pp"].gt(0).sum())
)

print("\nKISUMU AND TURKANA")
display(
    endpoint_changes.loc[
        endpoint_changes["county"].isin(
            ["Kisumu", "Turkana"]
        )
    ].round(3)
)

print("\nCOMPLETE SAMPLE COMPARISON")
display(sensitivity_summary.round(3))

print("\nCOUNTIES WITHOUT A DECLINE")
display(
    endpoint_changes.loc[
        endpoint_changes["change_pp"].ge(0)
    ].round(3)
)

assert len(endpoint_changes) == 47

print("\nCELL 11 ENDPOINT VERIFICATION: COMPLETE")


2013–2022 ENDPOINT PARTICIPATION CHECK
Counties compared: 47
Counties with declining participation: 47
Counties with increasing participation: 0

KISUMU AND TURKANA


election_id,2013_GENERAL,2022_GENERAL,change_pp,county
county_key,,,,
KISUMU,89.79,70.95,-18.84,Kisumu
TURKANA,75.04,60.17,-14.87,Turkana



COMPLETE SAMPLE COMPARISON


,sample,counties,counties_declining,mean_decline_pp,median_decline_pp,weighted_2013_pct,weighted_2022_pct,weighted_decline_pp,registration_growth_pct,valid_vote_growth_pct
0,45-county three-election sample,45,45,19.358,20.54,84.895,64.111,20.783,53.363,15.818
1,47-county endpoint sample,47,47,19.252,20.29,84.934,64.257,20.677,53.690,16.273



COUNTIES WITHOUT A DECLINE


election_id,2013_GENERAL,2022_GENERAL,change_pp,county
county_key,,,,



CELL 11 ENDPOINT VERIFICATION: COMPLETE


The results establish something important: valid-vote participation declined in all 47 Kenyan counties between 2013 and 2022. This conclusion holds even after including Kisumu and Turkana, which were previously excluded from our three-election comparisons because of August 2017 discrepancies.

**The central narrative is now well supported:**

Kenya's registered electorate grew substantially between 2013 and 2022, but the number of valid presidential votes grew much more slowly. Consequently, valid-vote participation declined across every county.

In [46]:

# ================================================================
# NOTEBOOK 06 — CELL 12
# GENERATE PORTFOLIO REPORT AND PROJECT README
# ================================================================
# Creates two new Markdown files.
# Does not overwrite existing files or alter analytical data.
# ================================================================

from pathlib import Path
from textwrap import dedent

portfolio_path = (
    REPORTS_DIR /
    "kenya_electoral_participation_portfolio_report.md"
)

readme_path = PROJECT_ROOT / "README.md"

# Protect existing work.
for path in [portfolio_path, readme_path]:
    assert not path.exists(), (
        f"File already exists; review before proceeding: {path}"
    )

# Retrieve the validated 47-county figures.
endpoint_results = sensitivity_summary.loc[
    sensitivity_summary["counties"].eq(47)
].iloc[0]

longitudinal_results = sensitivity_summary.loc[
    sensitivity_summary["counties"].eq(45)
].iloc[0]

# Verify expected analytical coverage.
assert int(endpoint_results["counties"]) == 47
assert int(endpoint_results["counties_declining"]) == 47
assert int(longitudinal_results["counties"]) == 45

# ------------------------------------------------
# FINAL PORTFOLIO REPORT
# ------------------------------------------------

portfolio_report = f"""
# Kenya Electoral Participation Analytics
## County-Level Presidential Election Analysis, 2013–2022

### Executive Summary

This project examines presidential electoral participation across
Kenya's 47 counties, focusing on the relationship between voter
registration growth and valid-vote participation.

Between 2013 and 2022, valid-vote participation declined in all
47 counties.

Across the complete 47-county endpoint sample:

- Registered voters increased by
  **{endpoint_results['registration_growth_pct']:.2f}%**.
- Valid presidential votes increased by
  **{endpoint_results['valid_vote_growth_pct']:.2f}%**.
- Weighted valid-vote participation declined from
  **{endpoint_results['weighted_2013_pct']:.2f}%**
  to **{endpoint_results['weighted_2022_pct']:.2f}%**.
- The weighted participation decline was
  **{endpoint_results['weighted_decline_pp']:.2f}
  percentage points**.
- The unweighted mean county decline was
  **{endpoint_results['mean_decline_pp']:.2f}
  percentage points**.

These findings describe changes in electoral participation.
They do not establish the causes of those changes.

### 1. Research Objectives

The analysis addresses four questions:

1. How did valid-vote participation change between 2013 and 2022?
2. How did registration growth compare with valid-vote growth?
3. How did participation trends vary across counties?
4. How did turnout change between the August and October 2017
   presidential elections?

### 2. Data and Methodology

The analytical dataset contains 188 county-election records:
47 counties across four presidential electoral events.

The events are:

- 2013 general election
- August 2017 general election
- October 2017 fresh presidential election
- 2022 general election

The October 2017 election is treated as a separate electoral
event rather than an ordinary general election.

#### Key measure

Valid-vote participation is calculated as:

**Valid-vote participation (%) =
(valid presidential votes / registered voters) × 100**

This measure is distinct from turnout based on ballots cast.

#### Analytical samples

The direct 2013–2022 endpoint analysis uses all **47 counties**.

Comparisons requiring August 2017 valid-vote participation
use **45 counties**, excluding Kisumu and Turkana because of
unresolved discrepancies between the ELOG and IEBC August 2017
valid-vote figures.

The same 45-county restriction is used for the August–October
2017 turnout comparison.

### 3. Electoral Participation, 2013–2022

Weighted valid-vote participation declined from
**{endpoint_results['weighted_2013_pct']:.2f}%** in 2013 to
**{endpoint_results['weighted_2022_pct']:.2f}%** in 2022.

All 47 counties recorded a decline.

The mean county decline was
**{endpoint_results['mean_decline_pp']:.2f} percentage points**.

The previously prepared three-election visualisation uses the
45-county comparable sample and should be interpreted on that basis.

![General election participation: 45-county sample](
figures/01_general_election_valid_vote_participation.png
)

### 4. County-Level Differences

Participation declined in every county between 2013 and 2022,
although the magnitude of decline varied geographically.

Kisumu's valid-vote participation fell from 89.79% to 70.95%,
a decline of 18.84 percentage points.

Turkana's participation fell from 75.04% to 60.17%,
a decline of 14.87 percentage points.

The existing county-ranking figure uses the 45-county
three-election sample.

![County participation declines: 45-county sample](
figures/02_county_participation_declines_2013_2022.png
)

### 5. Registration Growth Versus Valid-Vote Growth

Across all 47 counties between 2013 and 2022:

- Registration increased by
  **{endpoint_results['registration_growth_pct']:.2f}%**.
- Valid votes increased by
  **{endpoint_results['valid_vote_growth_pct']:.2f}%**.

Registration therefore expanded substantially faster than
the number of valid presidential votes.

This divergence explains the decline in the ratio of valid
votes to registered voters.

The existing growth visualisation uses the 45-county sample,
for which registration increased by
**{longitudinal_results['registration_growth_pct']:.2f}%**
and valid votes by
**{longitudinal_results['valid_vote_growth_pct']:.2f}%**.

![Registration and valid-vote growth: 45-county sample](
figures/03_registration_vs_valid_vote_growth_2013_2022.png
)

### 6. August–October 2017 Turnout

The August 2017 general election and October 2017 fresh
presidential election are analysed as distinct events.

Among the 45 counties with reconciled comparable turnout
figures, every county recorded a decline between the two events.

The unweighted mean county turnout decline was
approximately **39.09 percentage points**.

The comparison is descriptive and does not, by itself,
explain the political or behavioural causes of the change.

![August–October 2017 county turnout comparison](
figures/04_august_october_2017_county_turnout_comparison.png
)

![August–October 2017 turnout decline distribution](
figures/05_august_october_2017_turnout_decline_distribution.png
)

### 7. Source Reconciliation and Assurance

The analysis uses ELOG-derived county figures, with additional
reconciliation against published IEBC materials and a
2022 secondary transcription.

**2013:** County-level registration figures were reconciled
against an IEBC source. A difference of 2,637 voters remains
between the extracted county registration sum and the cited
national registration benchmark.

**August 2017:** Registration figures matched across all
47 counties. Kisumu and Turkana had unresolved valid-vote
differences and were excluded from analyses requiring those
August 2017 values.

**October 2017:** County registration and valid-vote figures
matched the IEBC comparison source across all 47 counties.

**2022:** ELOG registration and reported valid-vote totals
matched the Takwimu transcription for all 47 counties.

Two internal candidate-sum exceptions remain in the
Takwimu transcription:

- Kilifi: candidate-vote sum 10 below reported valid votes.
- West Pokot: candidate-vote sum 100 below reported valid votes.

The original signed IEBC Form 34C has not been independently
authenticated for this exercise. The 2022 assurance level is
therefore **secondary-source reconciled**, not officially verified.

### 8. Sensitivity Analysis

The original 45-county three-election sample and the complete
47-county endpoint sample produce closely aligned results.

| Measure | 45 counties | 47 counties |
|---|---:|---:|
| Counties with declining participation | 45 | 47 |
| Mean county decline (pp) | {longitudinal_results['mean_decline_pp']:.2f} | {endpoint_results['mean_decline_pp']:.2f} |
| Weighted decline (pp) | {longitudinal_results['weighted_decline_pp']:.2f} | {endpoint_results['weighted_decline_pp']:.2f} |
| Registration growth (%) | {longitudinal_results['registration_growth_pct']:.2f} | {endpoint_results['registration_growth_pct']:.2f} |
| Valid-vote growth (%) | {longitudinal_results['valid_vote_growth_pct']:.2f} | {endpoint_results['valid_vote_growth_pct']:.2f} |

Including Kisumu and Turkana does not change the direction
of the central 2013–2022 finding.

### 9. Limitations

- The analysis is descriptive, not causal.
- Valid-vote participation must not be described as
  ballots-cast turnout.
- Some historical source discrepancies remain unresolved.
- The 2022 reconciliation uses a secondary transcription
  attributed to Form 34C rather than a directly authenticated
  copy of the original signed declaration.
- The 45-county and 47-county samples must not be conflated.
- County-level aggregation cannot establish individual
  voter motivations or behaviour.

### 10. Conclusion

Between 2013 and 2022, Kenya's registered electorate grew
considerably faster than the number of valid presidential votes.

Valid-vote participation declined in all 47 counties,
indicating that the pattern was geographically widespread
rather than limited to a small group of counties.

The project demonstrates how transparent source
reconciliation, sample selection, reproducible calculations
and visual analysis can support a more reliable understanding
of electoral participation.

### Reproducibility

The project contains six sequential Jupyter notebooks,
intermediate and processed datasets, reconciliation registers,
validation summaries and five publication-ready figures.

All original source files are retained separately from
processed analytical outputs.
"""

# ------------------------------------------------
# PROJECT README
# ------------------------------------------------

readme = f"""
# Kenya Electoral Participation Analytics

A reproducible Python analysis of county-level presidential
electoral participation in Kenya across the 2013, August 2017,
October 2017 and 2022 elections.

## Project Overview

The study investigates changes in registered voters, valid
presidential votes and participation rates across Kenya's
47 counties.

### Headline Findings

- **47 of 47 counties** recorded declining valid-vote
  participation between 2013 and 2022.
- Registered voters increased by
  **{endpoint_results['registration_growth_pct']:.2f}%**.
- Valid presidential votes increased by
  **{endpoint_results['valid_vote_growth_pct']:.2f}%**.
- Weighted valid-vote participation fell by
  **{endpoint_results['weighted_decline_pp']:.2f}
  percentage points**.

## Methods

The project uses Python and pandas for data preparation,
reconciliation and analysis, with Matplotlib for visualisation.

Valid-vote participation is defined as:

`valid presidential votes / registered voters * 100`

It is not equivalent to ballots-cast turnout.

## Project Structure

```text
Kenya Electoral Participation Analytics Project/
├── data/
│   ├── Raw/
│   │   ├── elections/
│   │   └── reference/
│   ├── Interim/
│   └── processed/
├── documentation/
├── notebooks/
│   ├── 01_elog_data_extraction_validation.ipynb
│   ├── 02_official_source_reconciliation.ipynb
│   ├── 03_election_data_preparation.ipynb
│   ├── 04_electoral_participation_analysis.ipynb
│   ├── 05_electoral_participation_visualisation_reporting.ipynb
│   └── 06_final_project_synthesis_source_verification.ipynb
├── reports/
│   ├── figures/
│   └── kenya_electoral_participation_portfolio_report.md
└── README.md
```

## Analysis Workflow

1. Extract and validate ELOG county election results.
2. Reconcile historical figures against IEBC materials.
3. Prepare the integrated analytical dataset.
4. Analyse participation trends and county differences.
5. Produce and verify five analytical visualisations.
6. Reconcile 2022 figures and perform sensitivity checks.

## Analytical Coverage

The direct 2013–2022 endpoint comparison covers all
47 counties.

Comparisons requiring August 2017 figures use 45 counties
because Kisumu and Turkana have unresolved August 2017
source discrepancies.

October 2017 is treated as a distinct fresh presidential
election, not an ordinary general election.

## Data Quality and Source Assurance

The project retains documented discrepancies instead of
silently correcting source figures.

For 2022, ELOG county registration and reported valid-vote
figures match the Takwimu transcription in all 47 counties.
The original signed Form 34C has not been independently
authenticated.

Detailed reconciliation and validation records are
available in `documentation/`.

## Main Report

[Read the portfolio report](
reports/kenya_electoral_participation_portfolio_report.md
)

## Visualisations

Five verified figures are available in `reports/figures/`.

## Reproducing the Analysis

Run the six notebooks in numerical order from the
`notebooks/` directory.

The notebooks expect the project folder structure shown
above and the original source files under `data/Raw/`.

The analysis requires a Python environment with the
libraries imported by the notebooks, including pandas,
NumPy and Matplotlib.

## Interpretation

This is a descriptive electoral participation study.
It does not identify the causes of changes in voting
behaviour or establish causal relationships.
"""

# ------------------------------------------------
# WRITE OUTPUTS
# ------------------------------------------------

portfolio_path.write_text(
    dedent(portfolio_report).strip() + "\n",
    encoding="utf-8"
)

readme_path.write_text(
    dedent(readme).strip() + "\n",
    encoding="utf-8"
)

# ------------------------------------------------
# VERIFY OUTPUTS
# ------------------------------------------------

print("FINAL REPORTING OUTPUTS")
print("=" * 65)

for path in [portfolio_path, readme_path]:
    assert path.is_file()
    assert path.stat().st_size > 1000

    print(f"\nSaved: {path}")
    print(f"Size: {path.stat().st_size:,} bytes")

print("\nFIGURE REFERENCES")
for chart in expected_charts:
    assert (FIGURES_DIR / chart).is_file()
    assert chart in portfolio_report
    print("Verified:", chart)

print("\nCELL 12 REPORT GENERATION: PASSED")


FINAL REPORTING OUTPUTS

Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\reports\kenya_electoral_participation_portfolio_report.md
Size: 7,871 bytes

Saved: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\README.md
Size: 3,605 bytes

FIGURE REFERENCES
Verified: 01_general_election_valid_vote_participation.png
Verified: 02_county_participation_declines_2013_2022.png
Verified: 03_registration_vs_valid_vote_growth_2013_2022.png
Verified: 04_august_october_2017_county_turnout_comparison.png
Verified: 05_august_october_2017_turnout_decline_distribution.png

CELL 12 REPORT GENERATION: PASSED


### Publication-readiness audit
This cell checks the generated Markdown files, verifies the image links and identifies the Python libraries imported across all six notebooks.

It is read-only.

In [49]:

# ================================================================
# NOTEBOOK 06 — CELL 13
# PUBLICATION-READINESS AUDIT
# ================================================================
# Purpose:
# 1. Verify Markdown image links.
# 2. Check key reporting statements.
# 3. Identify notebook dependencies.
# 4. Confirm expected project files.
#
# Read-only: no files created or modified.
# ================================================================

import ast
import json
import re
from pathlib import Path

# ------------------------------------------------
# STEP 1: LOAD GENERATED DOCUMENTS
# ------------------------------------------------

report_text = portfolio_path.read_text(
    encoding="utf-8"
)

readme_text = readme_path.read_text(
    encoding="utf-8"
)

# ------------------------------------------------
# STEP 2: VERIFY MARKDOWN IMAGE LINKS
# ------------------------------------------------

image_links = re.findall(
    r"!\[[^\]]*\]\(\s*([^)]+?)\s*\)",
    report_text,
    flags=re.MULTILINE
)

image_audit = []

for link in image_links:
    clean_link = re.sub(r"\s+", "", link)
    image_path = REPORTS_DIR / clean_link

    image_audit.append({
        "image_link": clean_link,
        "exists": image_path.is_file()
    })

image_audit_df = pd.DataFrame(image_audit)

print("MARKDOWN IMAGE LINK AUDIT")
print("=" * 65)
display(image_audit_df)

# ------------------------------------------------
# STEP 3: VERIFY REPORTING STATEMENTS
# ------------------------------------------------

statement_checks = {
    "47-county endpoint coverage": "47 counties" in report_text,
    "45-county longitudinal coverage": "45 counties" in report_text,
    "Valid-vote participation definition":
        "Valid-vote participation" in report_text,
    "Kisumu exception documented": "Kisumu" in report_text,
    "Turkana exception documented": "Turkana" in report_text,
    "2022 secondary reconciliation":
        "secondary-source reconciled" in report_text.lower(),
    "Source limitations section":
        "### 9. Limitations" in report_text,
    "README reproducibility section":
        "## Reproducing the Analysis" in readme_text
}

statement_audit_df = pd.DataFrame([
    {
        "check": name,
        "passed": result
    }
    for name, result in statement_checks.items()
])

print("\nREPORTING STATEMENT AUDIT")
display(statement_audit_df)

# ------------------------------------------------
# STEP 4: IDENTIFY NOTEBOOK IMPORTS
# ------------------------------------------------

notebook_files = sorted(
    (PROJECT_ROOT / "notebooks").glob("0*.ipynb")
)

import_records = []

for notebook_path in notebook_files:

    notebook_data = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for cell in notebook_data.get("cells", []):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        try:
            syntax_tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in ast.walk(syntax_tree):

            if isinstance(node, ast.Import):
                for alias in node.names:
                    import_records.append({
                        "notebook": notebook_path.name,
                        "library": alias.name.split(".")[0]
                    })

            elif isinstance(node, ast.ImportFrom):
                if node.module:
                    import_records.append({
                        "notebook": notebook_path.name,
                        "library": node.module.split(".")[0]
                    })

imports_df = pd.DataFrame(import_records)

print("\nNOTEBOOK LIBRARY INVENTORY")

if not imports_df.empty:
    display(
        imports_df.drop_duplicates()
        .sort_values(["library", "notebook"])
    )
else:
    print("No imports identified.")

# ------------------------------------------------
# STEP 5: FINAL AUDIT
# ------------------------------------------------

assert len(notebook_files) == 6
assert len(image_audit_df) == 5
assert image_audit_df["exists"].all()
assert statement_audit_df["passed"].all()

print("\nPUBLICATION-READINESS AUDIT: PASSED")
print("Notebooks found:", len(notebook_files))
print("Image links verified:", len(image_audit_df))
print("Reporting checks passed:", len(statement_audit_df))


MARKDOWN IMAGE LINK AUDIT


,image_link,exists
0,figures/01_general_election_valid_vote_partici...,True
1,figures/02_county_participation_declines_2013_...,True
2,figures/03_registration_vs_valid_vote_growth_2...,True
3,figures/04_august_october_2017_county_turnout_...,True
4,figures/05_august_october_2017_turnout_decline...,True



REPORTING STATEMENT AUDIT


,check,passed
0,47-county endpoint coverage,True
1,45-county longitudinal coverage,True
2,Valid-vote participation definition,True
3,Kisumu exception documented,True
4,Turkana exception documented,True
5,2022 secondary reconciliation,True
6,Source limitations section,True
7,README reproducibility section,True



NOTEBOOK LIBRARY INVENTORY


,notebook,library
17,04_exploratory_electoral_participation_analysi...,matplotlib
31,05_electoral_participation_visualisation_repor...,matplotlib
16,04_exploratory_electoral_participation_analysi...,numpy
30,05_electoral_participation_visualisation_repor...,numpy
54,06_final_project_synthesis_source_verification...,numpy
2,01_elog_data_extraction_validation.ipynb,pandas
6,02_official_source_reconciliation.ipynb,pandas
9,03_election_data_preparation.ipynb,pandas
15,04_exploratory_electoral_participation_analysi...,pandas
29,05_electoral_participation_visualisation_repor...,pandas



PUBLICATION-READINESS AUDIT: PASSED
Notebooks found: 6
Image links verified: 5
Reporting checks passed: 8


### Prepare reproducibility files
This cell creates a requirements.txt file and a .gitignore file. It also checks the actual notebook filenames so we can avoid publishing an inaccurate project structure in the README.

In [52]:

# ================================================================
# NOTEBOOK 06 — CELL 14
# GITHUB REPRODUCIBILITY PREPARATION
# ================================================================
# Creates:
# 1. requirements.txt
# 2. .gitignore
#
# Does not modify datasets, notebooks or existing files.
# ================================================================

from pathlib import Path
import importlib.metadata as metadata

requirements_path = PROJECT_ROOT / "requirements.txt"
gitignore_path = PROJECT_ROOT / ".gitignore"

# ------------------------------------------------
# STEP 1: IDENTIFY INSTALLED PACKAGE VERSIONS
# ------------------------------------------------

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "pypdf",
    "jupyterlab"
]

requirements = []

print("INSTALLED PACKAGE VERSIONS")
print("=" * 65)

for package in packages:
    try:
        version = metadata.version(package)
        requirements.append(f"{package}=={version}")
        print(f"{package}: {version}")

    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# ------------------------------------------------
# STEP 2: CREATE REQUIREMENTS FILE
# ------------------------------------------------

if requirements_path.exists():
    print("\nExisting requirements.txt preserved.")
else:
    requirements_path.write_text(
        "\n".join(requirements) + "\n",
        encoding="utf-8"
    )
    print("\nCreated:", requirements_path)

# ------------------------------------------------
# STEP 3: CREATE GITIGNORE
# ------------------------------------------------

gitignore_content = """
# Python cache
__pycache__/
*.py[cod]

# Virtual environments
.venv/
venv/
env/

# Jupyter checkpoints
.ipynb_checkpoints/

# IDE settings
.vscode/
.idea/

# Operating system files
.DS_Store
Thumbs.db

# Temporary files
*.tmp
*.bak

# Local secrets and credentials
.env
.env.*
!.env.example
"""

if gitignore_path.exists():
    print("Existing .gitignore preserved.")
else:
    gitignore_path.write_text(
        gitignore_content.strip() + "\n",
        encoding="utf-8"
    )
    print("Created:", gitignore_path)

# ------------------------------------------------
# STEP 4: INSPECT ACTUAL NOTEBOOK FILENAMES
# ------------------------------------------------

print("\nACTUAL NOTEBOOK FILENAMES")
print("=" * 65)

for notebook_path in notebook_files:
    print(notebook_path.name)

# ------------------------------------------------
# STEP 5: VERIFY OUTPUTS
# ------------------------------------------------

assert requirements_path.is_file()
assert gitignore_path.is_file()

print("\nREQUIREMENTS FILE CONTENTS")
print(requirements_path.read_text(encoding="utf-8"))

print("\nCELL 14 REPRODUCIBILITY PREPARATION: PASSED")


INSTALLED PACKAGE VERSIONS
pandas: 2.1.4
numpy: 1.26.4
matplotlib: 3.8.0
pypdf: 7.0.0
jupyterlab: 4.0.11

Created: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\requirements.txt
Created: C:\Users\user\Downloads\Kenya Electoral Participation Analytics Project\.gitignore

ACTUAL NOTEBOOK FILENAMES
01_elog_data_extraction_validation.ipynb
02_official_source_reconciliation.ipynb
03_election_data_preparation.ipynb
04_exploratory_electoral_participation_analysis.ipynb
05_electoral_participation_visualisation_reporting.ipynb
06_final_project_synthesis_source_verification.ipynb

REQUIREMENTS FILE CONTENTS
pandas==2.1.4
numpy==1.26.4
matplotlib==3.8.0
pypdf==7.0.0
jupyterlab==4.0.11


CELL 14 REPRODUCIBILITY PREPARATION: PASSED


### GitHub repository audit
This cell will:
1. Inventory the project files and their sizes.
2. Identify unusually large files.
3. Check the source and data directories.
4. Identify notebook outputs that may unnecessarily increase repository size.
5. Compare the README's notebook references against the actual filenames.
It is read-only and does not modify any existing work.

In [55]:

# ================================================================
# NOTEBOOK 06 — CELL 15
# GITHUB REPOSITORY PUBLICATION AUDIT
# ================================================================
# Read-only audit of project structure, file sizes,
# notebook outputs and README filename accuracy.
# ================================================================

from pathlib import Path
import pandas as pd
import json
import re

# ------------------------------------------------
# STEP 1: INVENTORY PROJECT FILES
# ------------------------------------------------

file_records = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative_path = path.relative_to(PROJECT_ROOT)

    # Ignore local environments and temporary folders
    excluded_parts = {
        ".git", ".venv", "venv", "__pycache__",
        ".ipynb_checkpoints"
    }

    if any(part in excluded_parts for part in relative_path.parts):
        continue

    file_records.append({
        "relative_path": relative_path.as_posix(),
        "folder": relative_path.parts[0],
        "extension": path.suffix.lower(),
        "size_mb": path.stat().st_size / (1024 ** 2)
    })

file_inventory = pd.DataFrame(file_records)

print("PROJECT FILE INVENTORY")
print("=" * 65)

print("Total files:", len(file_inventory))
print(
    "Total size (MB):",
    round(file_inventory["size_mb"].sum(), 2)
)

# ------------------------------------------------
# STEP 2: DIRECTORY SUMMARY
# ------------------------------------------------

folder_summary = (
    file_inventory.groupby("folder")
    .agg(
        files=("relative_path", "count"),
        size_mb=("size_mb", "sum")
    )
    .sort_values("size_mb", ascending=False)
)

print("\nPROJECT FOLDER SUMMARY")
display(folder_summary.round(2))

# ------------------------------------------------
# STEP 3: LARGEST FILES
# ------------------------------------------------

print("\n15 LARGEST PROJECT FILES")

display(
    file_inventory.nlargest(
        15, "size_mb"
    ).round(2)
)

# ------------------------------------------------
# STEP 4: POTENTIALLY LARGE FILES
# ------------------------------------------------

large_files = file_inventory.loc[
    file_inventory["size_mb"] >= 25
]

print("\nFILES >= 25 MB")

if large_files.empty:
    print("None identified.")
else:
    display(large_files.round(2))

# ------------------------------------------------
# STEP 5: SOURCE AND DATA INVENTORY
# ------------------------------------------------

data_files = file_inventory.loc[
    file_inventory["relative_path"].str.startswith(
        "data/"
    )
]

print("\nDATA FILES BY EXTENSION")

display(
    data_files.groupby("extension")
    .agg(
        files=("relative_path", "count"),
        size_mb=("size_mb", "sum")
    )
    .round(2)
)

# ------------------------------------------------
# STEP 6: NOTEBOOK OUTPUT SIZE
# ------------------------------------------------

notebook_records = []

for notebook_path in notebook_files:

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    output_count = sum(
        len(cell.get("outputs", []))
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    )

    notebook_records.append({
        "notebook": notebook_path.name,
        "size_mb": notebook_path.stat().st_size / (1024 ** 2),
        "stored_outputs": output_count
    })

notebook_audit = pd.DataFrame(notebook_records)

print("\nNOTEBOOK OUTPUT AUDIT")
display(notebook_audit.round(2))

# ------------------------------------------------
# STEP 7: README NOTEBOOK REFERENCE AUDIT
# ------------------------------------------------

actual_names = {
    path.name for path in notebook_files
}

readme_names = set(
    re.findall(
        r"0[1-6]_[A-Za-z0-9_]+\.ipynb",
        readme_text
    )
)

print("\nREADME NOTEBOOK REFERENCES")

print("Actual notebooks:", len(actual_names))
print("README references:", len(readme_names))

print("\nMissing or incorrect README references:")
for name in sorted(actual_names - readme_names):
    print("NOT LISTED:", name)

print("\nREADME references without matching files:")
for name in sorted(readme_names - actual_names):
    print("NOT FOUND:", name)

print("\nCELL 15 REPOSITORY AUDIT: COMPLETE")


PROJECT FILE INVENTORY
Total files: 52
Total size (MB): 72.24

PROJECT FOLDER SUMMARY


,files,size_mb
folder,,
data,19,68.00
notebooks,6,2.42
reports,7,1.71
documentation,16,0.07
PROJECT_BRIEF.pdf,1,0.04
README.md,1,0.00
.gitignore,1,0.00
requirements.txt,1,0.00



15 LARGEST PROJECT FILES


,relative_path,folder,extension,size_mb
19,data/raw/reference/IEBC_2017_Presidential_Decl...,data,.pdf,27.50
16,data/raw/reference/IEBC_2013_General_Election_...,data,.pdf,25.99
20,data/raw/reference/IEBC_2022_Post_Election_Eva...,data,.pdf,6.76
17,data/raw/reference/IEBC_2017_Post_Election_Eva...,data,.pdf,3.14
18,data/raw/reference/IEBC_2017_Post_Election_Eva...,data,.pdf,3.14
43,notebooks/05_electoral_participation_visualisa...,notebooks,.ipynb,0.97
42,notebooks/04_exploratory_electoral_participati...,notebooks,.ipynb,0.73
49,reports/figures/03_registration_vs_valid_vote_...,reports,.png,0.43
50,reports/figures/04_august_october_2017_county_...,reports,.png,0.36
48,reports/figures/02_county_participation_declin...,reports,.png,0.35



FILES >= 25 MB


,relative_path,folder,extension,size_mb
16,data/raw/reference/IEBC_2013_General_Election_...,data,.pdf,25.99
19,data/raw/reference/IEBC_2017_Presidential_Decl...,data,.pdf,27.50



DATA FILES BY EXTENSION


,files,size_mb
extension,,
.csv,9,0.49
.pdf,10,67.51



NOTEBOOK OUTPUT AUDIT


,notebook,size_mb,stored_outputs
0,01_elog_data_extraction_validation.ipynb,0.07,24
1,02_official_source_reconciliation.ipynb,0.15,24
2,03_election_data_preparation.ipynb,0.27,87
3,04_exploratory_electoral_participation_analysi...,0.73,121
4,05_electoral_participation_visualisation_repor...,0.97,45
5,06_final_project_synthesis_source_verification...,0.24,76



README NOTEBOOK REFERENCES
Actual notebooks: 6
README references: 6

Missing or incorrect README references:
NOT LISTED: 04_exploratory_electoral_participation_analysis.ipynb

README references without matching files:
NOT FOUND: 04_electoral_participation_analysis.ipynb

CELL 15 REPOSITORY AUDIT: COMPLETE


### Prepare the GitHub repository
This cell makes three controlled changes:
- Updates the existing .gitignore to exclude source PDFs without deleting them.
- Corrects the README's notebook filename.
- Adds reproducibility and source-document guidance to the README.
It creates backup copies before modifying either file and leaves all analytical datasets and notebooks untouched.

In [58]:

# ================================================================
# NOTEBOOK 06 — CELL 16
# GITHUB REPOSITORY FINAL PREPARATION
# ================================================================
# Updates:
#   README.md
#   .gitignore
#
# Creates backups of both files.
# Does not delete raw data or modify notebooks.
# ================================================================

from pathlib import Path
import re
import shutil

readme_path = PROJECT_ROOT / "README.md"
gitignore_path = PROJECT_ROOT / ".gitignore"

readme_backup = PROJECT_ROOT / "README.pre_publication.bak"
gitignore_backup = PROJECT_ROOT / ".gitignore.pre_publication.bak"

# ------------------------------------------------
# STEP 1: SAFETY CHECKS AND BACKUPS
# ------------------------------------------------

assert readme_path.is_file()
assert gitignore_path.is_file()

assert not readme_backup.exists(), (
    "README backup already exists. Review before rerunning."
)
assert not gitignore_backup.exists(), (
    "Gitignore backup already exists. Review before rerunning."
)

shutil.copy2(readme_path, readme_backup)
shutil.copy2(gitignore_path, gitignore_backup)

print("BACKUPS CREATED")
print(readme_backup.name)
print(gitignore_backup.name)

# ------------------------------------------------
# STEP 2: CORRECT NOTEBOOK FILENAME
# ------------------------------------------------

readme_content = readme_path.read_text(encoding="utf-8")

old_name = "04_electoral_participation_analysis.ipynb"
new_name = "04_exploratory_electoral_participation_analysis.ipynb"

assert old_name in readme_content

readme_content = readme_content.replace(
    old_name,
    new_name
)

# ------------------------------------------------
# STEP 3: ADD REPRODUCIBILITY INSTRUCTIONS
# ------------------------------------------------

reproducibility_section = """
## Environment Setup

The project was developed using Python with the package
versions recorded in `requirements.txt`.

To create a local environment:

```bash
python -m venv .venv
```

Activate the environment on Windows:

```powershell
.venv\\Scripts\\Activate.ps1
```

Install the required packages:

```bash
python -m pip install -r requirements.txt
```

Start JupyterLab:

```bash
jupyter lab
```

Run the notebooks in numerical order, beginning with
Notebook 01.

**Note:** The notebooks use local project paths and expect
the original source documents to be available in the
appropriate `data/raw/` directories. Some source PDFs
are not included in the public repository.

## Source Documents and Redistribution

The project draws on ELOG electoral reports, IEBC
publications and a 2022 secondary transcription attributed
to IEBC Form 34C.

Original reference PDFs are excluded from the public
repository pending confirmation of redistribution rights.

The analysis retains documented reconciliation differences,
including unresolved August 2017 figures for Kisumu and
Turkana.

The 2022 ELOG and Takwimu county registration and reported
valid-vote totals match across all 47 counties. This
constitutes secondary-source corroboration, not independent
authentication of the original signed Form 34C.

Users seeking to reproduce the extraction stages should
obtain the original source documents from their publishers
and review the source references in the project notebooks.
"""

assert "## Environment Setup" not in readme_content

readme_content += "\n" + reproducibility_section.strip() + "\n"

# ------------------------------------------------
# STEP 4: EXCLUDE ORIGINAL SOURCE PDFS FROM GIT
# ------------------------------------------------

gitignore_content = gitignore_path.read_text(
    encoding="utf-8"
)

pdf_exclusion = """
# Original source PDFs are retained locally.
# Excluded from GitHub pending redistribution review.
data/raw/**/*.pdf
data/Raw/**/*.pdf

# Publication-preparation backups
*.pre_publication.bak
"""

gitignore_content += "\n" + pdf_exclusion.strip() + "\n"

# ------------------------------------------------
# STEP 5: SAVE UPDATES
# ------------------------------------------------

readme_path.write_text(
    readme_content,
    encoding="utf-8"
)

gitignore_path.write_text(
    gitignore_content,
    encoding="utf-8"
)

# ------------------------------------------------
# STEP 6: VERIFY CHANGES
# ------------------------------------------------

updated_readme = readme_path.read_text(
    encoding="utf-8"
)

updated_gitignore = gitignore_path.read_text(
    encoding="utf-8"
)

assert new_name in updated_readme
assert old_name not in updated_readme

assert "## Environment Setup" in updated_readme
assert "## Source Documents and Redistribution" in updated_readme

assert "data/raw/**/*.pdf" in updated_gitignore
assert "data/Raw/**/*.pdf" in updated_gitignore

print("\nGITHUB PREPARATION CHECKS")
print("=" * 65)

print("README filename corrected: PASS")
print("Environment instructions added: PASS")
print("Source limitations documented: PASS")
print("Original PDF exclusion added: PASS")
print("Existing analytical files preserved: PASS")

print("\nCELL 16 GITHUB PREPARATION: PASSED")


BACKUPS CREATED
README.pre_publication.bak
.gitignore.pre_publication.bak

GITHUB PREPARATION CHECKS
README filename corrected: PASS
Environment instructions added: PASS
Source limitations documented: PASS
Original PDF exclusion added: PASS
Existing analytical files preserved: PASS

CELL 16 GITHUB PREPARATION: PASSED


### Final GitHub publication audit
This cell checks the .gitignore rules, identifies files eligible for publication, and checks for potentially sensitive files. It does not initialise Git, stage files, or publish anything.

In [61]:

# ================================================================
# NOTEBOOK 06 — CELL 17
# FINAL GITHUB PUBLICATION AUDIT
# ================================================================
# Read-only:
# - No files created or modified
# - No Git commands executed
# - No changes to analytical outputs
# ================================================================

from pathlib import Path
import subprocess
import pandas as pd

# ------------------------------------------------
# STEP 1: CHECK GIT STATUS
# ------------------------------------------------

print("GIT REPOSITORY STATUS")
print("=" * 65)

git_check = subprocess.run(
    ["git", "rev-parse", "--is-inside-work-tree"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

if git_check.returncode == 0:
    print("Git repository detected.")
    print("Inside work tree:", git_check.stdout.strip())

    git_status = subprocess.run(
        ["git", "status", "--short"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    print("\nCurrent Git status:")
    print(git_status.stdout or "No changes reported.")

else:
    print("Git repository not yet initialised.")
    print("This is expected if GitHub setup has not started.")

# ------------------------------------------------
# STEP 2: VERIFY PDF EXCLUSION RULES
# ------------------------------------------------

gitignore_text = (
    PROJECT_ROOT / ".gitignore"
).read_text(encoding="utf-8")

pdf_rules_present = (
    "data/raw/**/*.pdf" in gitignore_text
    and "data/Raw/**/*.pdf" in gitignore_text
)

print("\nPDF EXCLUSION CONFIGURATION")
print("=" * 65)
print("PDF exclusion rules present:", pdf_rules_present)

# ------------------------------------------------
# STEP 3: INVENTORY SOURCE PDFS
# ------------------------------------------------

pdf_files = sorted(
    path for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".pdf"
    and "data" in path.relative_to(PROJECT_ROOT).parts
)

print("\nSOURCE PDF INVENTORY")
print("=" * 65)

for path in pdf_files:
    print(path.relative_to(PROJECT_ROOT))

print("\nSource PDFs found:", len(pdf_files))

# ------------------------------------------------
# STEP 4: CHECK PUBLICATION FILES
# ------------------------------------------------

required_publication_files = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / ".gitignore",
    portfolio_path,
    PROCESSED_DIR / "kenya_county_election_analytics.csv"
]

publication_checks = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.is_file()
    }
    for path in required_publication_files
])

print("\nESSENTIAL PUBLICATION FILES")
display(publication_checks)

# ------------------------------------------------
# STEP 5: IDENTIFY POTENTIAL SECRET FILES
# ------------------------------------------------

sensitive_names = {
    ".env",
    "credentials.json",
    "secrets.json",
    "service_account.json",
    "id_rsa",
    "id_ed25519"
}

sensitive_files = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
    and path.name.lower() in sensitive_names
]

print("\nPOTENTIAL SECRET FILES")

if sensitive_files:
    for path in sensitive_files:
        print("REVIEW:", path)
else:
    print("No common secret filenames identified.")

# ------------------------------------------------
# STEP 6: FINAL CHECKS
# ------------------------------------------------

assert pdf_rules_present
assert publication_checks["exists"].all()

print("\nCELL 17 PUBLICATION AUDIT: PASSED")
print("Git repository ready for staging review.")


GIT REPOSITORY STATUS
Git repository not yet initialised.
This is expected if GitHub setup has not started.

PDF EXCLUSION CONFIGURATION
PDF exclusion rules present: True

SOURCE PDF INVENTORY
data\raw\elections\elog\ELOG_2013_Presidential_General_Election.pdf
data\raw\elections\elog\ELOG_2017_August_Presidential_General_Election.pdf
data\raw\elections\elog\ELOG_2017_October_Fresh_Presidential_Election.pdf
data\raw\elections\elog\ELOG_2022_Presidential_General_Election.pdf
data\raw\reference\IEBC_2013_General_Election_Results.pdf
data\raw\reference\IEBC_2017_Post_Election_Evaluation.pdf
data\raw\reference\IEBC_2017_Post_Election_Evaluation_Report.pdf
data\raw\reference\IEBC_2017_Presidential_Declaration_Results.pdf
data\raw\reference\IEBC_2022_Post_Election_Evaluation_Report.pdf
data\raw\reference\kenya-2022-elections-final-report.pdf
data\raw\reference\Kenya_Gazette_2022_Presidential_Declaration.pdf

Source PDFs found: 11

ESSENTIAL PUBLICATION FILES


,file,exists
0,README.md,True
1,requirements.txt,True
2,.gitignore,True
3,reports\kenya_electoral_participation_portfoli...,True
4,data\processed\kenya_county_election_analytics...,True



POTENTIAL SECRET FILES
No common secret filenames identified.

CELL 17 PUBLICATION AUDIT: PASSED
Git repository ready for staging review.


### Final pre-publication checks
Before initialising Git, let's verify three things:
1. All six notebook filenames match the README.
2. The notebooks do not contain obvious credential assignments or tokens.
3. The .gitignore rules will exclude the source PDFs as intended.
The following cell is read-only. It does not create a repository or stage files.

In [64]:

# ================================================================
# NOTEBOOK 06 — CELL 18
# FINAL PRE-PUBLICATION VERIFICATION
# ================================================================

from pathlib import Path
import json
import re
import shutil
import subprocess
import pandas as pd

# ------------------------------------------------
# STEP 1: VERIFY NOTEBOOK REFERENCES
# ------------------------------------------------

actual_notebooks = sorted(
    path.name
    for path in (PROJECT_ROOT / "notebooks").glob("*.ipynb")
)

readme_content = (
    PROJECT_ROOT / "README.md"
).read_text(encoding="utf-8")

notebook_checks = pd.DataFrame([
    {
        "notebook": name,
        "listed_in_readme": name in readme_content
    }
    for name in actual_notebooks
])

print("NOTEBOOK REFERENCE VERIFICATION")
print("=" * 65)
display(notebook_checks)

# ------------------------------------------------
# STEP 2: SCAN NOTEBOOK SOURCE FOR POSSIBLE SECRETS
# ------------------------------------------------

# This is a heuristic scan, not a security guarantee.
# It reports only locations, not suspected secret values.

patterns = {
    "possible_password": r"(?i)\bpassword\s*=\s*['\"]",
    "possible_api_key": r"(?i)\bapi[_-]?key\s*=\s*['\"]",
    "possible_token": r"(?i)\b(?:access_token|auth_token)\s*=\s*['\"]",
    "private_key": r"-----BEGIN (?:RSA |OPENSSH )?PRIVATE KEY-----",
}

findings = []

for notebook_path in (
    PROJECT_ROOT / "notebooks"
).glob("*.ipynb"):

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for cell_number, cell in enumerate(
        notebook.get("cells", []), start=1
    ):
        source = "".join(cell.get("source", []))

        for label, pattern in patterns.items():
            if re.search(pattern, source):
                findings.append({
                    "notebook": notebook_path.name,
                    "cell": cell_number,
                    "finding": label
                })

print("\nPOTENTIAL CREDENTIAL REVIEW")

if findings:
    display(pd.DataFrame(findings))
else:
    print("No matches in the notebook source scan.")

# ------------------------------------------------
# STEP 3: TEST GITIGNORE RULES
# ------------------------------------------------

print("\nGITIGNORE RULE VERIFICATION")
print("=" * 65)

if shutil.which("git") is None:
    print("Git executable not found. Verify Git installation.")
else:
    pdf_paths = sorted(
        path.relative_to(PROJECT_ROOT).as_posix()
        for path in (PROJECT_ROOT / "data").rglob("*.pdf")
    )

    git_check = subprocess.run(
        ["git", "check-ignore", "--no-index", "--stdin"],
        input="\n".join(pdf_paths) + "\n",
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    ignored = set(git_check.stdout.splitlines())

    ignore_checks = pd.DataFrame([
        {
            "pdf": path,
            "excluded_by_gitignore": path in ignored
        }
        for path in pdf_paths
    ])

    display(ignore_checks)

# ------------------------------------------------
# STEP 4: SUMMARY
# ------------------------------------------------

print("\nFINAL PRE-PUBLICATION SUMMARY")

print(
    "Notebook references:",
    notebook_checks["listed_in_readme"].sum(),
    "/",
    len(notebook_checks)
)

print("Potential credential matches:", len(findings))

assert notebook_checks["listed_in_readme"].all()

if shutil.which("git") is not None:
    assert ignore_checks["excluded_by_gitignore"].all()

print("\nCELL 18 PRE-PUBLICATION VERIFICATION: PASSED")


NOTEBOOK REFERENCE VERIFICATION


,notebook,listed_in_readme
0,01_elog_data_extraction_validation.ipynb,True
1,02_official_source_reconciliation.ipynb,True
2,03_election_data_preparation.ipynb,True
3,04_exploratory_electoral_participation_analysi...,True
4,05_electoral_participation_visualisation_repor...,True
5,06_final_project_synthesis_source_verification...,True



POTENTIAL CREDENTIAL REVIEW
No matches in the notebook source scan.

GITIGNORE RULE VERIFICATION


,pdf,excluded_by_gitignore
0,data/raw/elections/elog/ELOG_2013_Presidential...,False
1,data/raw/elections/elog/ELOG_2017_August_Presi...,False
2,data/raw/elections/elog/ELOG_2017_October_Fres...,False
3,data/raw/elections/elog/ELOG_2022_Presidential...,False
4,data/raw/reference/IEBC_2013_General_Election_...,False
5,data/raw/reference/IEBC_2017_Post_Election_Eva...,False
6,data/raw/reference/IEBC_2017_Post_Election_Eva...,False
7,data/raw/reference/IEBC_2017_Presidential_Decl...,False
8,data/raw/reference/IEBC_2022_Post_Election_Eva...,False
9,data/raw/reference/Kenya_Gazette_2022_Presiden...,False



FINAL PRE-PUBLICATION SUMMARY
Notebook references: 6 / 6
Potential credential matches: 0


AssertionError: 

**18a Correct and verify PDF exclusions**

Let's first update .gitignore, then initialise Git locally and verify the exclusions.

Initialising Git is safe: it creates local version-control metadata but does not upload anything to GitHub, stage files, or create a commit.

In [67]:

# ================================================================
# NOTEBOOK 06 — CELL 18A
# FIX AND VERIFY GITIGNORE PDF EXCLUSIONS
# ================================================================
# Actions:
# 1. Add a clear PDF exclusion rule.
# 2. Initialise Git locally if necessary.
# 3. Verify that all 11 source PDFs are ignored.
#
# Does NOT stage, commit or publish any files.
# ================================================================

from pathlib import Path
import subprocess
import shutil
import pandas as pd

gitignore_path = PROJECT_ROOT / ".gitignore"

# ------------------------------------------------
# STEP 1: VERIFY GIT INSTALLATION
# ------------------------------------------------

assert shutil.which("git") is not None, (
    "Git is not installed or is not available on PATH."
)

# ------------------------------------------------
# STEP 2: ADD EXPLICIT PDF EXCLUSION RULE
# ------------------------------------------------

gitignore_content = gitignore_path.read_text(
    encoding="utf-8"
)

pdf_rule = "/data/raw/**/*.pdf"

if pdf_rule not in gitignore_content.splitlines():
    with gitignore_path.open("a", encoding="utf-8") as file:
        file.write(
            "\n# Exclude original electoral source PDFs\n"
            + pdf_rule + "\n"
        )

print("PDF exclusion rule verified:", pdf_rule)

# ------------------------------------------------
# STEP 3: INITIALISE LOCAL GIT REPOSITORY
# ------------------------------------------------

git_directory = PROJECT_ROOT / ".git"

if not git_directory.exists():

    result = subprocess.run(
        ["git", "init"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    print("\nGIT INITIALISATION")
    print(result.stdout.strip())

    assert result.returncode == 0, result.stderr

else:
    print("\nExisting Git repository detected.")

# ------------------------------------------------
# STEP 4: TEST ALL SOURCE PDFS
# ------------------------------------------------

pdf_files = sorted(
    path for path in (PROJECT_ROOT / "data").rglob("*")
    if path.is_file() and path.suffix.lower() == ".pdf"
)

pdf_paths = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in pdf_files
]

result = subprocess.run(
    ["git", "check-ignore", "--no-index", "--stdin"],
    input="\n".join(pdf_paths) + "\n",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

ignored_paths = set(result.stdout.splitlines())

ignore_checks = pd.DataFrame([
    {
        "pdf": path,
        "excluded_by_gitignore": path in ignored_paths
    }
    for path in pdf_paths
])

print("\nSOURCE PDF EXCLUSION RESULTS")
display(ignore_checks)

# ------------------------------------------------
# STEP 5: FINAL VERIFICATION
# ------------------------------------------------

print("\nSUMMARY")
print("PDF files found:", len(pdf_paths))
print(
    "PDF files excluded:",
    int(ignore_checks["excluded_by_gitignore"].sum())
)

assert result.returncode in (0, 1), result.stderr
assert len(pdf_paths) == 11
assert ignore_checks["excluded_by_gitignore"].all()

print("\nCELL 18A: PASSED")
print("All source PDFs are excluded from Git staging.")


PDF exclusion rule verified: /data/raw/**/*.pdf

GIT INITIALISATION
Initialized empty Git repository in C:/Users/user/Downloads/Kenya Electoral Participation Analytics Project/.git/

SOURCE PDF EXCLUSION RESULTS


,pdf,excluded_by_gitignore
0,data/raw/elections/elog/ELOG_2013_Presidential...,False
1,data/raw/elections/elog/ELOG_2017_August_Presi...,False
2,data/raw/elections/elog/ELOG_2017_October_Fres...,False
3,data/raw/elections/elog/ELOG_2022_Presidential...,False
4,data/raw/reference/IEBC_2013_General_Election_...,False
5,data/raw/reference/IEBC_2017_Post_Election_Eva...,False
6,data/raw/reference/IEBC_2017_Post_Election_Eva...,False
7,data/raw/reference/IEBC_2017_Presidential_Decl...,False
8,data/raw/reference/IEBC_2022_Post_Election_Eva...,False
9,data/raw/reference/kenya-2022-elections-final-...,False



SUMMARY
PDF files found: 11
PDF files excluded: 0


AssertionError: 

**18B: Diagnose and fix PDF exclusions**

This cell will replace the previous PDF exclusion patterns with a simpler rule:

In [71]:

# ================================================================
# NOTEBOOK 06 — CELL 18B
# DIAGNOSE AND CORRECT GITIGNORE CONFIGURATION
# ================================================================
# Does not delete source files, stage files or publish.
# Updates only .gitignore.
# ================================================================

from pathlib import Path
import subprocess
import pandas as pd

gitignore_path = PROJECT_ROOT / ".gitignore"

# ------------------------------------------------
# STEP 1: INSPECT EXISTING GITIGNORE
# ------------------------------------------------

current_content = gitignore_path.read_text(
    encoding="utf-8"
)

print("CURRENT GITIGNORE")
print("=" * 65)
print(current_content)

# ------------------------------------------------
# STEP 2: ADD A SIMPLE DIRECTORY EXCLUSION
# ------------------------------------------------

rule = "/data/raw/"

if rule not in current_content.splitlines():

    with gitignore_path.open(
        "a",
        encoding="utf-8"
    ) as file:
        file.write(
            "\n# Keep original source files local\n"
            + rule + "\n"
        )

print("\nAdded or confirmed rule:", rule)

# ------------------------------------------------
# STEP 3: CHECK GIT REPOSITORY ROOT
# ------------------------------------------------

root_check = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("\nGIT REPOSITORY ROOT")
print(root_check.stdout.strip())

assert root_check.returncode == 0, root_check.stderr

# ------------------------------------------------
# STEP 4: TEST EACH PDF INDIVIDUALLY
# ------------------------------------------------

pdf_files = sorted(
    path for path in (PROJECT_ROOT / "data").rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".pdf"
)

results = []

for path in pdf_files:

    relative_path = path.relative_to(
        PROJECT_ROOT
    ).as_posix()

    check = subprocess.run(
        [
            "git",
            "check-ignore",
            "-v",
            "--no-index",
            relative_path
        ],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )

    results.append({
        "pdf": relative_path,
        "ignored": check.returncode == 0,
        "matching_rule": check.stdout.strip(),
        "error": check.stderr.strip()
    })

ignore_results = pd.DataFrame(results)

print("\nINDIVIDUAL PDF IGNORE TEST")
display(ignore_results)

# ------------------------------------------------
# STEP 5: VERIFY RAW DIRECTORY IS EXCLUDED
# ------------------------------------------------

print("\nFINAL SUMMARY")
print("PDFs found:", len(ignore_results))
print("PDFs ignored:", int(ignore_results["ignored"].sum()))

assert len(ignore_results) == 11
assert ignore_results["ignored"].all(), (
    "Some PDFs are still not ignored. "
    "Inspect matching_rule and error columns."
)

print("\nCELL 18B: PASSED")
print("All original source PDFs are excluded.")


CURRENT GITIGNORE
# Python cache
__pycache__/
*.py[cod]

# Virtual environments
.venv/
venv/
env/

# Jupyter checkpoints
.ipynb_checkpoints/

# IDE settings
.vscode/
.idea/

# Operating system files
.DS_Store
Thumbs.db

# Temporary files
*.tmp
*.bak

# Local secrets and credentials
.env
.env.*
!.env.example

# Original source PDFs are retained locally.
# Excluded from GitHub pending redistribution review.
data/raw/**/*.pdf
data/Raw/**/*.pdf

# Publication-preparation backups
*.pre_publication.bak

# Exclude original electoral source PDFs
/data/raw/**/*.pdf


Added or confirmed rule: /data/raw/

GIT REPOSITORY ROOT
C:/Users/user/Downloads/Kenya Electoral Participation Analytics Project

INDIVIDUAL PDF IGNORE TEST


,pdf,ignored,matching_rule,error
0,data/raw/elections/elog/ELOG_2013_Presidential...,True,.gitignore:42:/data/raw/\tdata/raw/elections/e...,
1,data/raw/elections/elog/ELOG_2017_August_Presi...,True,.gitignore:42:/data/raw/\tdata/raw/elections/e...,
2,data/raw/elections/elog/ELOG_2017_October_Fres...,True,.gitignore:42:/data/raw/\tdata/raw/elections/e...,
3,data/raw/elections/elog/ELOG_2022_Presidential...,True,.gitignore:42:/data/raw/\tdata/raw/elections/e...,
4,data/raw/reference/IEBC_2013_General_Election_...,True,.gitignore:42:/data/raw/\tdata/raw/reference/I...,
5,data/raw/reference/IEBC_2017_Post_Election_Eva...,True,.gitignore:42:/data/raw/\tdata/raw/reference/I...,
6,data/raw/reference/IEBC_2017_Post_Election_Eva...,True,.gitignore:42:/data/raw/\tdata/raw/reference/I...,
7,data/raw/reference/IEBC_2017_Presidential_Decl...,True,.gitignore:42:/data/raw/\tdata/raw/reference/I...,
8,data/raw/reference/IEBC_2022_Post_Election_Eva...,True,.gitignore:42:/data/raw/\tdata/raw/reference/I...,
9,data/raw/reference/kenya-2022-elections-final-...,True,.gitignore:42:/data/raw/\tdata/raw/reference/k...,



FINAL SUMMARY
PDFs found: 11
PDFs ignored: 11

CELL 18B: PASSED
All original source PDFs are excluded.


Before creating the first commit, let's inspect the exact list of files Git considers eligible for tracking.

We will also check whether the README's project structure and reproducibility instructions accurately describe the files that will be available publicly.

**Git publication manifest**

In [74]:

# ================================================================
# NOTEBOOK 06 — CELL 19
# REVIEW GIT PUBLICATION MANIFEST
# ================================================================
# Read-only: does not stage, commit or upload files.
# ================================================================

from pathlib import Path
import subprocess
import pandas as pd

# ------------------------------------------------
# STEP 1: LIST FILES ELIGIBLE FOR GIT
# ------------------------------------------------

result = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard", "-z"],
    cwd=PROJECT_ROOT,
    capture_output=True
)

assert result.returncode == 0, (
    result.stderr.decode("utf-8", errors="replace")
)

eligible_files = [
    path.decode("utf-8", errors="replace")
    for path in result.stdout.split(b"\0")
    if path
]

manifest = pd.DataFrame([
    {
        "file": relative_path,
        "extension": Path(relative_path).suffix.lower(),
        "size_mb": round(
            (PROJECT_ROOT / relative_path).stat().st_size
            / (1024 ** 2),
            3
        )
    }
    for relative_path in eligible_files
])

print("GIT PUBLICATION MANIFEST")
print("=" * 65)

if not manifest.empty:
    display(manifest.sort_values("file").reset_index(drop=True))

    print("\nSUMMARY")
    print("Eligible files:", len(manifest))
    print("Total size (MB):", round(manifest["size_mb"].sum(), 2))

    print("\nFILES BY EXTENSION")
    display(
        manifest.groupby("extension")
        .agg(
            files=("file", "count"),
            size_mb=("size_mb", "sum")
        )
        .round(3)
    )
else:
    print("No untracked, non-ignored files found.")

# ------------------------------------------------
# STEP 2: VERIFY RAW SOURCES ARE NOT INCLUDED
# ------------------------------------------------

raw_files = [
    path for path in eligible_files
    if path.lower().startswith("data/raw/")
]

print("\nRAW SOURCE EXCLUSION CHECK")
print("Raw files eligible for Git:", len(raw_files))

if raw_files:
    for path in raw_files:
        print("REVIEW:", path)

# ------------------------------------------------
# STEP 3: CHECK ESSENTIAL ANALYTICAL FILES
# ------------------------------------------------

required_files = [
    "README.md",
    "requirements.txt",
    ".gitignore",
    "data/processed/kenya_county_election_analytics.csv",
    "reports/kenya_electoral_participation_portfolio_report.md"
]

required_files += [
    f"notebooks/{path.name}"
    for path in notebook_files
]

required_files += [
    f"reports/figures/{filename}"
    for filename in expected_charts
]

publication_checks = pd.DataFrame([
    {
        "file": path,
        "eligible_for_git": path in eligible_files
    }
    for path in required_files
])

print("\nESSENTIAL FILE CHECK")
display(publication_checks)

# ------------------------------------------------
# STEP 4: FINAL VALIDATION
# ------------------------------------------------

assert not raw_files, (
    "Raw source files unexpectedly eligible for Git."
)

assert publication_checks["eligible_for_git"].all(), (
    "Some essential files are missing from the Git manifest."
)

print("\nCELL 19 PUBLICATION MANIFEST: PASSED")
print("Ready for final staging review.")


GIT PUBLICATION MANIFEST


,file,extension,size_mb
0,.gitignore,,0.001
1,PROJECT_BRIEF.pdf,.pdf,0.037
2,README.md,.md,0.005
3,data/interim/election_county_integrated_workin...,.csv,0.068
4,data/interim/elog_county_election_results.csv,.csv,0.038
5,data/interim/elog_main_general_elections.csv,.csv,0.028
6,data/interim/iebc_2013_county_official.csv,.csv,0.002
7,data/interim/iebc_2017_august_county_official.csv,.csv,0.001
8,data/interim/iebc_2017_october_county_official...,.csv,0.001
9,data/interim/takwimu_2022_county_presidential.csv,.csv,0.004



SUMMARY
Eligible files: 41
Total size (MB): 4.5

FILES BY EXTENSION


,files,size_mb
extension,,
,1,0.001
.csv,22,0.243
.ipynb,6,2.491
.md,5,0.027
.pdf,1,0.037
.png,5,1.698
.txt,1,0.000



RAW SOURCE EXCLUSION CHECK
Raw files eligible for Git: 0

ESSENTIAL FILE CHECK


,file,eligible_for_git
0,README.md,True
1,requirements.txt,True
2,.gitignore,True
3,data/processed/kenya_county_election_analytics...,True
4,reports/kenya_electoral_participation_portfoli...,True
5,notebooks/01_elog_data_extraction_validation.i...,True
6,notebooks/02_official_source_reconciliation.ipynb,True
7,notebooks/03_election_data_preparation.ipynb,True
8,notebooks/04_exploratory_electoral_participati...,True
9,notebooks/05_electoral_participation_visualisa...,True



CELL 19 PUBLICATION MANIFEST: PASSED
Ready for final staging review.


### Final README correction and publication review
This cell updates the README and prints the Git manifest for review. It does not stage or commit files.

In [77]:

# ================================================================
# NOTEBOOK 06 — CELL 20
# FINAL README CORRECTION AND PUBLICATION REVIEW
# ================================================================

from pathlib import Path
import subprocess

readme_path = PROJECT_ROOT / "README.md"

readme_content = readme_path.read_text(
    encoding="utf-8"
)

# ------------------------------------------------
# STEP 1: CORRECT RAW-SOURCE AVAILABILITY WORDING
# ------------------------------------------------

old_text = (
    "Original reference PDFs are excluded from the public\n"
    "repository pending confirmation of redistribution rights."
)

new_text = (
    "The entire `data/raw/` directory is excluded from the\n"
    "public repository pending review of source redistribution\n"
    "rights. This includes original reference PDFs and the\n"
    "2022 secondary-source CSV transcription.\n\n"
    "Intermediate datasets, the final processed analytical\n"
    "dataset, reconciliation tables and validation outputs\n"
    "remain available in the repository.\n\n"
    "Notebooks 01–03 require the original source files for\n"
    "complete end-to-end reproduction. Notebooks 04–06 can\n"
    "be examined alongside the published analytical outputs."
)

if old_text in readme_content:
    readme_content = readme_content.replace(
        old_text,
        new_text
    )
    readme_path.write_text(
        readme_content,
        encoding="utf-8"
    )
    print("README source availability wording updated.")
elif new_text in readme_content:
    print("README already contains the updated wording.")
else:
    print("README wording not matched. No changes made.")
    print("Review the Source Documents and Redistribution section.")

# ------------------------------------------------
# STEP 2: VERIFY README AND PROJECT BRIEF
# ------------------------------------------------

print("\nFINAL PUBLICATION CHECK")
print("=" * 65)

print("README exists:", readme_path.exists())
print(
    "Project brief exists:",
    (PROJECT_ROOT / "PROJECT_BRIEF.pdf").exists()
)

# ------------------------------------------------
# STEP 3: DISPLAY ELIGIBLE GIT FILES
# ------------------------------------------------

result = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

assert result.returncode == 0, result.stderr

eligible_files = result.stdout.splitlines()

print("\nFILES ELIGIBLE FOR FIRST COMMIT")
print("=" * 65)

for filename in eligible_files:
    print(filename)

print("\nTotal eligible files:", len(eligible_files))

# ------------------------------------------------
# STEP 4: CHECK FOR UNEXPECTED RAW FILES
# ------------------------------------------------

raw_files = [
    filename for filename in eligible_files
    if filename.lower().startswith("data/raw/")
]

assert not raw_files, (
    "Raw source files unexpectedly included."
)

print("\nCELL 20: PUBLICATION REVIEW COMPLETE")
print("No files have been staged or committed.")


README source availability wording updated.

FINAL PUBLICATION CHECK
README exists: True
Project brief exists: True

FILES ELIGIBLE FOR FIRST COMMIT
.gitignore
PROJECT_BRIEF.pdf
README.md
data/interim/election_county_integrated_working.csv
data/interim/elog_county_election_results.csv
data/interim/elog_main_general_elections.csv
data/interim/iebc_2013_county_official.csv
data/interim/iebc_2017_august_county_official.csv
data/interim/iebc_2017_october_county_official.csv
data/interim/takwimu_2022_county_presidential.csv
data/processed/kenya_county_election_analytics.csv
documentation/election_data_methodology_and_validation.md
documentation/notebook01_validation_summary.csv
documentation/notebook02_reconciliation_summary.csv
documentation/notebook04_analysis_summary.md
documentation/notebook04_key_findings.csv
documentation/notebook04_validation_summary.csv
documentation/notebook05_chart_inventory.csv
documentation/notebook05_validation_summary.csv
documentation/notebook06_2022_source_v